# Auto Align-VLM v3 — complete experiment & publication pipeline (Colab, A100)

**Auto Align-VLM: A Reinforcement Learning Framework for Mitigating Object Hallucination in Vision–Language Models**

This notebook runs the full study end-to-end and writes every figure, table and number the manuscript needs into **one Drive folder** (`…/auto-align-vlm_v3/publication/`). Nothing in the outputs is typed by hand: every table cell, figure and in-text number is computed from cached raw model outputs and collected in `publication/data/results_master.json`.

### What changed from v2 (and why it matters for the paper)
Reading the v2 code that produced the earlier results revealed four issues that a reviewer could find; v3 fixes all of them, so **all v2 numbers are superseded by this run**:

| v2 behaviour (as executed) | consequence | v3 |
|---|---|---|
| Quality term was `MockCider` (reference unigram recall), although the manuscript calls it "normalised CIDEr" | method description did not match the code | term is named and described exactly; CIDEr-D is not used because its length penalty zeroes paragraph-length captions |
| Base model cached with `lru_cache`, then wrapped in place by `get_peft_model` during training | the POPE "base" numbers were computed on a model object that already carried injected LoRA layers | every policy is loaded fresh; adapters merged into a private copy; a guard asserts the base has no LoRA modules |
| LoRA `target_modules` also matched q/k/v of the CLIP vision tower (2.36 M of 42.3 M trainable params) | manuscript said the vision encoder was frozen | LoRA restricted to the language model; asserted at training time |
| Compact CHAIR synonym map, 128-token captions, stratified (non-standard) image sample | absolute CHAIR not comparable with the literature; truncated captions in DPO pairs | official CHAIR (port verified to match the reference implementation exactly), 500 random val2014 images, 512-token captions, only complete samples in pairs |

### Experimental design (Q1-level evidence)
* **Main model**, 3 training seeds → mean ± SD; paired image-level bootstrap CIs, Wilcoxon and exact McNemar tests (Holm-corrected).
* **Matched controls** (same images, same samples, same schedule — only the preference signal changes): *length* (prefer the shortest caption — tests "it just says less") and *random* pairs.
* **Ablations**: fixed COCO-80 grounding vs caption-conditioned; no quality term (α = 0); λ ∈ {1, 2, 4}.
* **Baselines**: training-free VCD decoding, and Auto Align-VLM + VCD (complementarity).
* **RL iteration**: a second on-policy round (re-sample from the round-1 policy on new images).
* **Benchmarks**: CHAIR (with object recall and length), POPE (random/popular/adversarial), MME (capability preservation).
* **Reward validity**: agreement between the annotation-free reward and annotation-based CHAIR on held-out captions.
* **Measured compute** for every stage (GPU-hours, peak memory) → compute table.

### How to run (resource-efficient)
1. `Runtime → Change runtime type → A100`. Run cell 0–1 with `PROFILE = "smoke"` and **Run all**. The smoke run (~30–45 min) exercises every code path on ~24 images and writes to a separate `_smoke` folder. A bug costs minutes, not GPU-hours.
2. Set `PROFILE = "full"`, `RUN_TIERS = {1}`, run all. Tier 1 contains everything needed for the main claims.
3. If budget remains, set `RUN_TIERS = {1, 2}` and re-run: finished work is skipped automatically.
4. **After any disconnect**: re-run from the top. Data, samples, detections, pairs, adapters, checkpoints and every evaluation output are cached on Drive; an interrupted DPO run resumes from its last checkpoint.

**Rough planning estimates** (not measurements — the notebook prints *measured* projections after the first DPO run, and the paper reports only measured cost): first-session setup 20–30 min; base evaluation ~0.5–1 h; on-policy sampling + detection + scoring ~1.5–3 h; one DPO run ~3–5 h; one policy evaluation ~0.5–1 h. Tier 1 ≈ 6 DPO runs; Tier 2 adds 4 more and a second sampling round.


## 0 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1 · Configuration
The only cell you normally edit. It is saved with the results (`publication/data/run_config.json`).

In [ ]:
# ============================================================================
# 1) CONFIGURATION — the single source of truth for every setting in the paper
# ============================================================================
# Edit ONLY this cell. Every downstream number, table and figure is derived
# from this dictionary, and the dictionary itself is saved next to the results.
#
# PROFILE = "smoke": tiny end-to-end run (~25-40 min on A100). ALWAYS run it
#                    first in a fresh runtime: it exercises every code path
#                    (download, sampling, detection, rewards, DPO, CHAIR, POPE,
#                    MME, VCD, figures, tables) on a few images, so a bug costs
#                    minutes instead of GPU-hours. It writes to a separate
#                    Drive folder and never touches full results.
# PROFILE = "full" : the publication run.
# ============================================================================
from pathlib import Path
import os, json, copy

PROFILE = "full"            # "smoke" -> then switch to "full"

# Which experiment tiers to run (see the experiment registry, cell 12).
#   Tier 1 = required for the main claims (3-seed main model, matched controls,
#            key ablations, VCD baseline).  Tier 2 = sensitivity & iteration.
RUN_TIERS = {1, 2}

DRIVE_BASE = Path("/content/drive/MyDrive/ResearchforPub/auto-align-vlm-execution")
RUN_NAME   = "auto-align-vlm_v3" + ("_smoke" if PROFILE == "smoke" else "")
ROOT       = DRIVE_BASE / RUN_NAME           # persistent (Google Drive)
LOCAL      = Path("/content/aavlm_local")    # fast, ephemeral (Colab disk)

P = {  # all paths
    "root": ROOT,
    "cache": ROOT / "cache",            # data selections, samples, detections, scores, pairs
    "runs": ROOT / "runs",              # DPO adapters, checkpoints, training logs
    "eval": ROOT / "eval",              # raw model outputs + per-policy metrics
    "pub": ROOT / "publication",        # everything the paper needs, in one place
    "fig": ROOT / "publication" / "figures",
    "tab": ROOT / "publication" / "tables",
    "pubdata": ROOT / "publication" / "data",
    "logs": ROOT / "logs",
    "local": LOCAL,
    "pkgs": LOCAL / "python_packages",
    "models": LOCAL / "models",
    "images": LOCAL / "images",         # COCO jpgs actually used (restored from a Drive tar)
    "dl": LOCAL / "downloads",
    "hf_cache": LOCAL / "hf_cache",
    "nltk": LOCAL / "nltk_data",
}

_full = dict(
    # ---------------- data ----------------
    data_seed=2024,                 # fixes every image selection
    n_train=5000,                   # training images per on-policy round
    n_chair=500,                    # standard CHAIR protocol: 500 random COCO val2014 images
    pope_settings=["random", "popular", "adversarial"],
    pope_limit=None,                # None = all 3,000 questions per setting
    run_mme=True,                   # MME (perception + cognition) capability check
    mme_limit=None,
    # ---------------- prompts / decoding ----------------
    caption_prompt="Please describe this image in detail.",
    short_answer_suffix="Answer the question using a single word or phrase.",
    max_new_tokens_caption=512,     # long-form captions (standard CHAIR setting)
    require_finished=True,          # only complete (EOS-terminated) samples enter preference pairs
    max_new_tokens_short=4,
    sampling=dict(k=4, temperature=0.9, top_p=0.95),
    batch_caption=16,               # greedy captioning batch (images)
    batch_sample=8,                 # sampling batch (images; x k sequences)
    batch_short=32,                 # POPE / MME batch
    # ---------------- models ----------------
    vlm_repo="llava-hf/llava-1.5-7b-hf",
    detector_repo="IDEA-Research/grounding-dino-tiny",   # Grounding DINO Swin-T (OGC)
    matcher_repo="sentence-transformers/all-MiniLM-L6-v2",
    # ---------------- HAR ----------------
    detector=dict(box_threshold=0.30, text_threshold=0.25, keep_score=0.35,
                  max_phrases=40, batch=8),
    har=dict(alpha=1.0, lam=2.0, tau=0.50, min_margin=0.25),
    # ---------------- DPO ----------------
    train=dict(lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_on_vision=False,
               lr=1e-5, beta=0.3, epochs=2, per_device_bs=2, grad_accum=8,
               max_length=1280, logging_steps=5, save_steps=100, warmup_ratio=0.0),
    seeds=[42, 123, 456],
    # ---------------- baselines ----------------
    vcd=dict(alpha=1.0, beta=0.1, noise_step=500, seed=0),
    # ---------------- statistics / figures ----------------
    n_bootstrap=10000,
    fig_dpi=600,
)

_smoke_overrides = dict(
    n_train=24, n_chair=12, pope_limit=24, mme_limit=24,
    max_new_tokens_caption=128, require_finished=False, n_bootstrap=500,
    sampling=dict(k=4, temperature=0.9, top_p=0.95),
    train=dict(_full["train"], epochs=1, grad_accum=2, save_steps=5, logging_steps=1),
    seeds=[42, 123],                # two seeds so seed aggregation is exercised
    fig_dpi=200,
)

C = copy.deepcopy(_full)
if PROFILE == "smoke":
    C.update(copy.deepcopy(_smoke_overrides))
C["profile"] = PROFILE
C["run_tiers"] = sorted(RUN_TIERS)

for k, v in P.items():
    if k not in ("local",):
        Path(v).mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

print(f"PROFILE={PROFILE} | tiers={sorted(RUN_TIERS)}")
print("persistent root :", ROOT)
print("local scratch   :", LOCAL)


PROFILE=full | tiers=[1, 2]
persistent root : /content/drive/MyDrive/ResearchforPub/auto-align-vlm-execution/auto-align-vlm_v3
local scratch   : /content/aavlm_local


## 2 · Environment
Pinned to the exact library versions that ran v2 on Colab; Colab's CUDA torch is untouched. Grounding DINO uses its native Transformers port — no CUDA extension to compile.

In [ ]:
# ============================================================================
# 2) ENVIRONMENT — pinned libraries (no torch reinstall), NLP data, GPU check
# ============================================================================
# The library versions below are exactly the ones that ran your v2 pipeline
# successfully on Colab (transformers 4.57.6 / trl 0.29.1 / peft 0.19.1).
# They are installed into a LOCAL target directory with --no-deps so Colab's
# CUDA-enabled torch is never replaced.  Grounding DINO is used through its
# native Transformers port, so no CUDA extension has to be compiled.
import subprocess, sys, os, shutil, importlib, json, time, platform, urllib.request, zipfile

PINNED = [
    "transformers==4.57.6", "tokenizers>=0.22,<=0.23.0", "accelerate==1.14.0",
    "peft==0.19.1", "trl==0.29.1", "datasets==3.6.0", "huggingface-hub==0.36.2",
    "sentence-transformers==3.4.1",
]
stamp = P["pkgs"] / ".installed.json"
if not (stamp.exists() and json.loads(stamp.read_text()).get("pins") == PINNED):
    shutil.rmtree(P["pkgs"], ignore_errors=True); P["pkgs"].mkdir(parents=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", str(P["pkgs"]),
                        "--no-deps", "--upgrade", *PINNED], text=True, capture_output=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode != 0:
        raise RuntimeError("pip install failed (see log above)")
    stamp.write_text(json.dumps({"pins": PINNED}))
if str(P["pkgs"]) not in sys.path:
    sys.path.insert(0, str(P["pkgs"]))

# torchao is not needed for bf16 LoRA-DPO; when present in the base image it can
# trigger a duplicate-operator registration crash inside peft. Make it unimportable.
for k in [k for k in list(sys.modules) if k == "torchao" or k.startswith("torchao.")]:
    del sys.modules[k]
sys.modules["torchao"] = None

os.environ.update({
    "TOKENIZERS_PARALLELISM": "false", "USE_TF": "0", "TRANSFORMERS_NO_TF": "1",
    "HF_HOME": str(P["hf_cache"]), "HF_DATASETS_CACHE": str(P["hf_cache"] / "datasets"),
    "NLTK_DATA": str(P["nltk"]), "WANDB_DISABLED": "true",
})
try:  # optional HF token (Colab secret "HF_TOKEN") — raises download rate limits
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

# ---- spaCy English model (used by the caption parser) ----
import spacy
try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

# ---- NLTK data (used by the official CHAIR implementation) ----
import nltk
P["nltk"].mkdir(parents=True, exist_ok=True)
nltk.data.path.insert(0, str(P["nltk"]))
_NLTK = {"tokenizers/punkt_tab": "punkt_tab",
         "taggers/averaged_perceptron_tagger_eng": "averaged_perceptron_tagger_eng",
         "corpora/wordnet": "wordnet", "corpora/omw-1.4": "omw-1.4"}
for res, pkg in _NLTK.items():
    try:
        nltk.data.find(res)
    except LookupError:
        ok = nltk.download(pkg, download_dir=str(P["nltk"]), quiet=True)
        if not ok:  # direct fallback
            sub = res.split("/")[0]; dst = P["nltk"] / sub; dst.mkdir(parents=True, exist_ok=True)
            zp = dst / f"{pkg}.zip"
            urllib.request.urlretrieve(
                f"https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/{res}.zip", zp)
            zipfile.ZipFile(zp).extractall(dst)

import torch, transformers, peft, trl, datasets as hfds
assert torch.cuda.is_available(), "Select a GPU runtime (A100 recommended)."
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GPU = torch.cuda.get_device_name(0)
ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda, "gpu": GPU,
       "gpu_mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
       "transformers": transformers.__version__, "peft": peft.__version__,
       "trl": trl.__version__, "datasets": hfds.__version__, "spacy": spacy.__version__,
       "nltk": nltk.__version__}
print(json.dumps(ENV, indent=1))
if "A100" not in GPU:
    print("WARNING: not an A100. Batch sizes in cell 1 assume ~80 GB; reduce them on smaller GPUs.")


{
 "python": "3.13.16",
 "torch": "2.14.1+cu130",
 "cuda": "13.0",
 "gpu": "NVIDIA A100-SXM4-80GB",
 "gpu_mem_gb": 79.3,
 "transformers": "4.57.6",
 "peft": "0.19.1",
 "trl": "0.29.1",
 "datasets": "3.6.0",
 "spacy": "3.8.18",
 "nltk": "3.9.1"
}


## 3 · Utilities
Resumable JSONL stores (one file per stage — fast on Drive), a compute ledger that times every stage, provenance.

In [ ]:
# ============================================================================
# 3) UTILITIES — resumable JSONL stores, compute ledger, seeding, provenance
# ============================================================================
import json, os, time, random, hashlib, gc, contextlib, datetime
from pathlib import Path
import numpy as np


def jdump(obj, path):
    """Atomic JSON write (a crash never leaves a half-written file)."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=_json_default)
    os.replace(tmp, path)


def jload(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path) as f:
        return json.load(f)


def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    if isinstance(o, set): return sorted(o)
    raise TypeError(type(o))


class JsonlStore:
    """Append-only, keyed JSONL file. One file per stage (Drive-friendly: no
    thousands of tiny files). Resume = skip keys already present."""

    def __init__(self, path, key="key"):
        self.path = Path(path); self.key = key
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._d = {}
        if self.path.exists():
            with open(self.path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:   # torn last line after a crash
                        continue
                    self._d[str(r[self.key])] = r

    def __contains__(self, k): return str(k) in self._d
    def __getitem__(self, k): return self._d[str(k)]
    def __len__(self): return len(self._d)
    def keys(self): return list(self._d.keys())
    def values(self): return list(self._d.values())
    def get(self, k, default=None): return self._d.get(str(k), default)

    def add_many(self, records):
        with open(self.path, "a") as f:
            for r in records:
                self._d[str(r[self.key])] = r
                f.write(json.dumps(r, ensure_ascii=False, default=_json_default) + "\n")
            f.flush(); os.fsync(f.fileno())


# ---------------------------------------------------------------- compute ledger
LEDGER = P["logs"] / "compute_ledger.jsonl"


@contextlib.contextmanager
def ledger(stage, **meta):
    """Records measured wall-clock time and peak GPU memory of every stage;
    the compute-cost table in the paper is built from these measurements."""
    import torch
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    status = "ok"
    try:
        yield
    except BaseException:
        status = "interrupted_or_failed"
        raise
    finally:
        rec = {"stage": stage, "seconds": round(time.time() - t0, 1), "status": status,
               "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)
               if torch.cuda.is_available() else None,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               "time": datetime.datetime.now().isoformat(timespec="seconds"), **meta}
        with open(LEDGER, "a") as f:
            f.write(json.dumps(rec) + "\n")
        print(f"[ledger] {stage}: {rec['seconds']/60:.1f} min, peak {rec['peak_gpu_gb']} GB ({status})")


def read_ledger():
    if not LEDGER.exists():
        return []
    return [json.loads(l) for l in open(LEDGER) if l.strip()]


def set_all_seeds(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_gpu(*objs):
    import torch
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def config_hash(d):
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:10]


def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


# Save the configuration + environment next to the results (provenance).
jdump({"config": C, "env": ENV, "config_hash": config_hash(C),
       "saved": datetime.datetime.now().isoformat(timespec="seconds")},
      P["pubdata"] / "run_config.json")
print("utilities ready | config hash", config_hash(C))


utilities ready | config hash 8553c4b43c


## 4 · Data
Downloads only the ~1–2 GB of COCO images actually used (v2 downloaded ~19 GB each session) and keeps them as tar files on Drive. Training images come from the Karpathy *train* split (train2014 only), evaluation images from val2014; an assertion guarantees no overlap.

In [ ]:
# ============================================================================
# 4) DATA — selections, annotations, POPE, MME, and a minimal image fetch
# ============================================================================
# Resource note: v2 downloaded the full COCO train2014+val2014 zips (~19 GB)
# every session. v3 fetches only the images the study uses (~1-2 GB), keeps
# them as tar files on Drive, and restores them in seconds in later sessions.
import zipfile, tarfile, io, urllib.request, urllib.error, concurrent.futures as cf
from PIL import Image

COCO_IMG_URL = "http://images.cocodataset.org/{split}/{fname}"
URLS = {
    "karpathy": "https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip",
    "coco_ann": "http://images.cocodataset.org/annotations/annotations_trainval2014.zip",
    "pope": "https://raw.githubusercontent.com/RUCAIBox/POPE/main/output/coco/coco_pope_{s}.json",
}


def download(url, dst, retries=4):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    for a in range(retries):
        try:
            tmp = dst.with_suffix(dst.suffix + ".part")
            urllib.request.urlretrieve(url, tmp); os.replace(tmp, dst); return dst
        except Exception as e:
            print(f"  download retry {a+1}/{retries} for {url}: {e}"); time.sleep(5 * (a + 1))
    raise RuntimeError(f"failed to download {url}")


def coco_fname(image_id, split):
    return f"COCO_{split}_{int(image_id):012d}.jpg"


def image_path(image_id, split):
    return P["images"] / split / coco_fname(image_id, split)


# ---------------------------------------------------------------- annotations
def prepare_annotations():
    """Karpathy references + compact COCO val2014 annotations, cached on Drive."""
    kp_index = P["cache"] / "karpathy_train.json"
    val_ann = P["cache"] / "coco_val2014_compact.json"
    if not kp_index.exists():
        z = download(URLS["karpathy"], P["dl"] / "caption_datasets.zip")
        with zipfile.ZipFile(z) as zf:
            name = [n for n in zf.namelist() if n.endswith("dataset_coco.json")][0]
            data = json.loads(zf.read(name))["images"]
        # Training pool = Karpathy 'train' split, train2014 images only (no 'restval',
        # whose val2014 images could overlap the POPE / CHAIR evaluation images).
        train = [{"image_id": im["cocoid"], "refs": [s["raw"].strip() for s in im["sentences"]]}
                 for im in data if im["split"] == "train" and "train2014" in im["filepath"]]
        jdump(train, kp_index)
    if not val_ann.exists():
        z = download(URLS["coco_ann"], P["dl"] / "annotations_trainval2014.zip")
        with zipfile.ZipFile(z) as zf:
            inst = json.loads(zf.read("annotations/instances_val2014.json"))
            caps = json.loads(zf.read("annotations/captions_val2014.json"))
        cat = {c["id"]: c["name"] for c in inst["categories"]}
        comp = {}
        for a in inst["annotations"]:
            comp.setdefault(str(a["image_id"]), {"cats": set(), "caps": []})["cats"].add(cat[a["category_id"]])
        for a in caps["annotations"]:
            comp.setdefault(str(a["image_id"]), {"cats": set(), "caps": []})["caps"].append(a["caption"])
        comp = {k: {"cats": sorted(v["cats"]), "caps": v["caps"]} for k, v in comp.items()}
        jdump(comp, val_ann)
        (P["dl"] / "annotations_trainval2014.zip").unlink(missing_ok=True)   # 241 MB freed
    return jload(kp_index), jload(val_ann)


def prepare_pope():
    out = {}
    for s in C["pope_settings"]:
        f = download(URLS["pope"].format(s=s), P["cache"] / "pope" / f"coco_pope_{s}.json")
        items = [json.loads(l) for l in open(f) if l.strip()]
        if C["pope_limit"]:
            # stratified subset keeps the yes/no balance in smoke runs
            yes = [x for x in items if x["label"] == "yes"]; no = [x for x in items if x["label"] == "no"]
            h = C["pope_limit"] // 2; items = yes[:h] + no[:h]
        out[s] = items
    return out


# ---------------------------------------------------------------- selections
def make_selections(kp_train, val_ann, pope):
    sel_path = P["cache"] / "selections.json"
    sel = jload(sel_path)
    if sel is not None:
        return sel
    rng = random.Random(C["data_seed"])
    # CHAIR: n random COCO val2014 images that carry instance annotations
    val_ids = sorted(int(k) for k, v in val_ann.items() if v["cats"])
    chair_ids = rng.sample(val_ids, C["n_chair"])
    # training rounds: disjoint random draws from the Karpathy train pool
    pool = [r["image_id"] for r in kp_train]
    rng.shuffle(pool)
    n = C["n_train"]
    sel = {"chair_ids": chair_ids, "train_r1": pool[:n], "train_r2": pool[n:2 * n],
           "pope_images": sorted({int(x["image"].split("_")[-1].split(".")[0])
                                  for s in pope.values() for x in s})}
    evalset = set(sel["chair_ids"]) | set(sel["pope_images"])
    assert not evalset & set(sel["train_r1"] + sel["train_r2"]), "train/eval image overlap!"
    jdump(sel, sel_path)
    return sel


# ---------------------------------------------------------------- image fetch
def _fetch_one(args):
    image_id, split = args
    dst = image_path(image_id, split)
    if dst.exists() and dst.stat().st_size > 0:
        return image_id, True
    url = COCO_IMG_URL.format(split=split, fname=coco_fname(image_id, split))
    for a in range(4):
        try:
            data = urllib.request.urlopen(url, timeout=60).read()
            Image.open(io.BytesIO(data)).verify()
            dst.parent.mkdir(parents=True, exist_ok=True)
            with open(dst, "wb") as f:
                f.write(data)
            return image_id, True
        except Exception:
            time.sleep(1 + 2 * a)
    return image_id, False


def ensure_images(ids, split, group):
    """Restore a group of images from its Drive tar, fetch anything missing,
    then refresh the tar so the next session restores everything instantly."""
    tar = P["cache"] / "images" / f"{group}.tar"
    (P["images"] / split).mkdir(parents=True, exist_ok=True)
    if tar.exists():
        with tarfile.open(tar) as tf:
            try:
                tf.extractall(P["images"], filter="data")
            except TypeError:          # Python < 3.12
                tf.extractall(P["images"])
    missing = [i for i in ids if not image_path(i, split).exists()]
    if missing:
        print(f"[images] fetching {len(missing)} {split} images for '{group}' ...")
        with cf.ThreadPoolExecutor(32) as ex:
            res = list(ex.map(_fetch_one, [(i, split) for i in missing]))
        failed = [i for i, ok in res if not ok]
        if failed:
            raise RuntimeError(f"{len(failed)} images failed to download, e.g. {failed[:5]}. "
                               "Re-run this cell; downloads resume.")
        tar.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(str(tar) + ".tmp", "w") as tf:
            for i in ids:
                tf.add(image_path(i, split), arcname=f"{split}/{coco_fname(i, split)}")
        os.replace(str(tar) + ".tmp", tar)
    print(f"[images] {group}: {len(ids)} ready")


# ---------------------------------------------------------------- MME
def prepare_mme():
    """MME (Fu et al.) from the lmms-lab mirror on the Hugging Face Hub.
    Saved once to Drive so later sessions load it without re-downloading."""
    if not C["run_mme"]:
        return None
    from datasets import load_dataset, load_from_disk
    disk = P["cache"] / "mme_hf"
    if disk.exists():
        ds = load_from_disk(str(disk))
    else:
        ds = load_dataset("lmms-lab/MME", split="test")
        ds.save_to_disk(str(disk))
    if C["mme_limit"]:
        # keep whole question pairs from several categories for the smoke test
        ds = ds.select(range(min(len(ds), C["mme_limit"])))
    return ds


def build_data():
    kp_train, val_ann = prepare_annotations()
    pope = prepare_pope()
    sel = make_selections(kp_train, val_ann, pope)
    refs = {r["image_id"]: r["refs"] for r in kp_train}
    D = {
        "val_ann": val_ann, "pope": pope, "sel": sel,
        "train_r1": [{"image_id": i, "path": str(image_path(i, "train2014")), "refs": refs[i]}
                     for i in sel["train_r1"]],
        "train_r2": [{"image_id": i, "path": str(image_path(i, "train2014")), "refs": refs[i]}
                     for i in sel["train_r2"]],
        "chair": [{"image_id": i, "path": str(image_path(i, "val2014"))} for i in sel["chair_ids"]],
    }
    for s, items in pope.items():
        for x in items:
            x["path"] = str(P["images"] / "val2014" / x["image"])
    return D


with ledger("data_preparation"):
    D = build_data()
    ensure_images(D["sel"]["chair_ids"], "val2014", "eval_chair")
    ensure_images(D["sel"]["pope_images"], "val2014", "eval_pope")
    ensure_images(D["sel"]["train_r1"], "train2014", "train_r1")
    MME = prepare_mme()
print({"train_r1": len(D["train_r1"]), "chair": len(D["chair"]),
       **{f"pope_{s}": len(v) for s, v in D["pope"].items()},
       "mme": (len(MME) if MME is not None else 0)})


[images] fetching 488 val2014 images for 'eval_chair' ...
[images] eval_chair: 500 ready
[images] fetching 492 val2014 images for 'eval_pope' ...
[images] eval_pope: 500 ready
[images] fetching 4991 train2014 images for 'train_r1' ...
[images] train_r1: 5000 ready


README.md:   0%|          | 0.00/493 [00:00<?, ?B/s]

data/test-00000-of-00004-a25dbe3b44c4fda(…):   0%|          | 0.00/34.9M [00:00<?, ?B/s]

data/test-00001-of-00004-7d22c7f1aba6fca(…):   0%|          | 0.00/110M [00:00<?, ?B/s]

data/test-00002-of-00004-594798fd3f5b029(…):   0%|          | 0.00/695M [00:00<?, ?B/s]

data/test-00003-of-00004-53ae1794f93b1e3(…):   0%|          | 0.00/24.7M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2374 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/2374 [00:00<?, ? examples/s]

[ledger] data_preparation: 2.7 min, peak 0.02 GB (ok)
{'train_r1': 5000, 'chair': 500, 'pope_random': 3000, 'pope_popular': 3000, 'pope_adversarial': 3000, 'mme': 2374}


## 5 · Models and decoding
Fresh, never-shared policy objects; batched left-padded generation; batched visual contrastive decoding (VCD). On a tiny test model, batched decoding reproduced per-sample decoding exactly and VCD with α = 0 reproduced greedy decoding token-for-token.

In [ ]:
# ============================================================================
# 5) MODELS & DECODING — clean policy loading, batched generation, VCD
# ============================================================================
# Correctness fix vs v2: v2 cached the base model with lru_cache and then
# wrapped that same object with get_peft_model during training, so every
# later "base" evaluation in the same process ran on a model that already
# carried injected LoRA layers. v3 never mutates a shared model: every policy
# is loaded fresh, adapters are merged into that private copy, and a guard
# asserts that the "base" policy contains no LoRA modules.
import torch
from concurrent.futures import ThreadPoolExecutor

MODEL_DIRS = {}
DEVICE_MAP = {"": 0} if torch.cuda.is_available() else {"": "cpu"}
POLICY_DTYPE = torch.bfloat16


def snapshot(repo):
    from huggingface_hub import snapshot_download
    d = P["models"] / repo.split("/")[-1]
    if not (d / "config.json").exists():
        snapshot_download(repo_id=repo, local_dir=str(d),
                          token=os.environ.get("HF_TOKEN") or None)
    return str(d)


def ensure_models():
    MODEL_DIRS["vlm"] = snapshot(C["vlm_repo"])
    MODEL_DIRS["detector"] = snapshot(C["detector_repo"])
    MODEL_DIRS["matcher"] = snapshot(C["matcher_repo"])
    return MODEL_DIRS


def load_processor():
    from transformers import AutoProcessor
    try:
        proc = AutoProcessor.from_pretrained(MODEL_DIRS["vlm"])
    except Exception as e:           # fast-tokenizer incompatibility seen on some Colab images
        print("[processor] default load failed, retrying with use_fast=False:", e)
        proc = AutoProcessor.from_pretrained(MODEL_DIRS["vlm"], use_fast=False)
    proc.tokenizer.padding_side = "left"   # required for batched decoder-only generation
    return proc


def has_lora(model):
    return any("lora_" in n for n, _ in model.named_modules())


def load_policy(spec):
    """spec = {"name": str, "adapters": [dir, ...]} -> fresh model with adapters merged.
    Adapters are merged in order (round-2 adapters are trained on top of round 1)."""
    from transformers import LlavaForConditionalGeneration
    kw = dict(device_map=DEVICE_MAP, attn_implementation="sdpa")
    try:
        model = LlavaForConditionalGeneration.from_pretrained(MODEL_DIRS["vlm"], dtype=POLICY_DTYPE, **kw)
    except TypeError:
        model = LlavaForConditionalGeneration.from_pretrained(MODEL_DIRS["vlm"], torch_dtype=POLICY_DTYPE, **kw)
    for a in spec.get("adapters", []):
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, a).merge_and_unload()
    model.eval()
    if not spec.get("adapters"):
        assert not has_lora(model), "base policy must not contain LoRA modules"
    model.generation_config.pad_token_id = PROC.tokenizer.pad_token_id
    return model


def caption_prompt():
    return f"USER: <image>\n{C['caption_prompt']} ASSISTANT:"


def short_prompt(question):
    q = question.strip()
    return f"USER: <image>\n{q}\n{C['short_answer_suffix']} ASSISTANT:"


def load_images(paths):
    def _open(p):
        with Image.open(p) as im:
            return im.convert("RGB")
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(_open, paths))


def _decode(gen_ids, tok):
    eos, pad = tok.eos_token_id, tok.pad_token_id
    out = []
    for row in gen_ids.tolist():
        n, fin = len(row), False
        for j, t in enumerate(row):
            if t == eos:
                n, fin = j, True; break
        while n > 0 and row[n - 1] == pad:
            n -= 1
        out.append({"text": tok.decode(row[:n], skip_special_tokens=True).strip(),
                    "n_tokens": n, "finished": fin})
    return out


@torch.no_grad()
def generate_batch(model, images, prompts, max_new_tokens, sample=False, k=1,
                   temperature=1.0, top_p=1.0, seed=None):
    """Batched generation. Returns one list (length k) of outputs per image."""
    inputs = PROC(images=images, text=prompts, return_tensors="pt", padding=True).to(model.device)
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
    if seed is not None:
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    gkw = dict(max_new_tokens=max_new_tokens, num_beams=1, pad_token_id=PROC.tokenizer.pad_token_id)
    if sample:
        gkw.update(do_sample=True, temperature=temperature, top_p=top_p, num_return_sequences=k)
    else:
        gkw.update(do_sample=False, temperature=None, top_p=None)
    out = model.generate(**inputs, **gkw)
    dec = _decode(out[:, inputs["input_ids"].shape[1]:], PROC.tokenizer)
    return [dec[i * k:(i + 1) * k] for i in range(len(images))]


# ------------------------------------------------------------------ VCD
def add_diffusion_noise(x, noise_step, generator):
    """Forward-diffusion corruption of the processed image, as in the official
    VCD implementation (Leng et al., CVPR 2024; 1000-step sigmoid beta schedule)."""
    num_steps = 1000
    betas = torch.sigmoid(torch.linspace(-6, 6, num_steps)) * (0.5e-2 - 1e-5) + 1e-5
    alphas_prod = torch.cumprod(1 - betas, dim=0)
    a = alphas_prod[noise_step].sqrt().item(); b = (1 - alphas_prod[noise_step]).sqrt().item()
    noise = torch.randn(x.shape, generator=generator, device=x.device, dtype=torch.float32)
    return (a * x.float() + b * noise).to(x.dtype)


@torch.no_grad()
def vcd_generate_batch(model, images, prompts, max_new_tokens, alpha, beta, noise_step, seed):
    """Greedy visual contrastive decoding with the adaptive plausibility constraint:
       logits_cd = (1+alpha) * l(v) - alpha * l(v'),  restricted to tokens with
       p(token | v) >= beta * max p(. | v).  Two KV caches are advanced in lockstep."""
    tok = PROC.tokenizer
    inputs = PROC(images=images, text=prompts, return_tensors="pt", padding=True).to(model.device)
    pv = inputs["pixel_values"].to(model.dtype)
    g = torch.Generator(device=pv.device).manual_seed(seed)
    pv_cd = add_diffusion_noise(pv, noise_step, g)
    ids, am = inputs["input_ids"], inputs["attention_mask"]
    pos = (am.long().cumsum(-1) - 1).clamp(min=0)
    o1 = model(input_ids=ids, attention_mask=am, pixel_values=pv, position_ids=pos, use_cache=True)
    o2 = model(input_ids=ids, attention_mask=am, pixel_values=pv_cd, position_ids=pos, use_cache=True)
    kv1, kv2 = o1.past_key_values, o2.past_key_values
    l1, l2 = o1.logits[:, -1, :].float(), o2.logits[:, -1, :].float()
    B = ids.shape[0]
    done = torch.zeros(B, dtype=torch.bool, device=ids.device)
    steps = []
    log_beta = float(np.log(beta))
    for _ in range(max_new_tokens):
        cutoff = l1.max(dim=-1, keepdim=True).values + log_beta
        cd = (1 + alpha) * l1 - alpha * l2
        cd = cd.masked_fill(l1 < cutoff, float("-inf"))
        nxt = cd.argmax(dim=-1)
        nxt = torch.where(done, torch.full_like(nxt, tok.pad_token_id), nxt)
        steps.append(nxt)
        done = done | (nxt == tok.eos_token_id)
        if bool(done.all()):
            break
        am = torch.cat([am, torch.ones((B, 1), dtype=am.dtype, device=am.device)], dim=1)
        p_new = (am.long().sum(-1, keepdim=True) - 1)
        o1 = model(input_ids=nxt[:, None], attention_mask=am, position_ids=p_new,
                   past_key_values=kv1, use_cache=True)
        o2 = model(input_ids=nxt[:, None], attention_mask=am, position_ids=p_new,
                   past_key_values=kv2, use_cache=True)
        kv1, kv2 = o1.past_key_values, o2.past_key_values
        l1, l2 = o1.logits[:, -1, :].float(), o2.logits[:, -1, :].float()
    gen = torch.stack(steps, dim=1)
    return [[d] for d in _decode(gen, tok)]


# ------------------------------------------------------------------ runners
def run_captions(model, records, store_path, decoding="greedy", bs=None, desc=""):
    """Greedy (or VCD) long-form captions for CHAIR. Resumable per batch."""
    store = JsonlStore(store_path, key="image_id")
    todo = [r for r in records if r["image_id"] not in store]
    bs = bs or C["batch_caption"]
    i = 0; t0 = time.time()
    while i < len(todo):
        batch = todo[i:i + bs]
        try:
            imgs = load_images([r["path"] for r in batch])
            prompts = [caption_prompt()] * len(batch)
            if decoding == "vcd":
                v = C["vcd"]
                outs = vcd_generate_batch(model, imgs, prompts, C["max_new_tokens_caption"],
                                          v["alpha"], v["beta"], v["noise_step"], v["seed"] + i)
            else:
                outs = generate_batch(model, imgs, prompts, C["max_new_tokens_caption"])
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache(); bs = max(1, bs // 2); print(f"  OOM -> batch {bs}"); continue
        store.add_many([{"image_id": r["image_id"], "caption": o[0]["text"],
                         "n_tokens": o[0]["n_tokens"], "finished": o[0]["finished"]}
                        for r, o in zip(batch, outs)])
        i += len(batch)
        if (i // bs) % 5 == 0 or i >= len(todo):
            print(f"  {desc} captions {len(store)}/{len(records)} ({(time.time()-t0)/60:.1f} min)")
    return {str(r["image_id"]): store[r["image_id"]] for r in records}


def run_sampling(model, records, store_path, seed_base):
    """k on-policy samples per training image (one prefill per image,
    num_return_sequences=k). Each batch has its own fixed seed -> resumable
    and reproducible."""
    s = C["sampling"]; store = JsonlStore(store_path, key="image_id")
    todo = [r for r in records if r["image_id"] not in store]
    bs = C["batch_sample"]; i = 0; t0 = time.time()
    while i < len(todo):
        batch = todo[i:i + bs]
        try:
            imgs = load_images([r["path"] for r in batch])
            outs = generate_batch(model, imgs, [caption_prompt()] * len(batch),
                                  C["max_new_tokens_caption"], sample=True, k=s["k"],
                                  temperature=s["temperature"], top_p=s["top_p"],
                                  seed=seed_base + int(batch[0]["image_id"]) % 100000)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache(); bs = max(1, bs // 2); print(f"  OOM -> batch {bs}"); continue
        store.add_many([{"image_id": r["image_id"], "samples": [x["text"] for x in o],
                         "n_tokens": [x["n_tokens"] for x in o],
                         "finished": [x["finished"] for x in o]} for r, o in zip(batch, outs)])
        i += len(batch)
        if (i // bs) % 10 == 0 or i >= len(todo):
            el = (time.time() - t0) / 60
            print(f"  sampled {len(store)}/{len(records)} images ({el:.1f} min, "
                  f"~{el / max(i,1) * (len(todo)-i):.0f} min left)")
    return store


def parse_yes_no(text):
    t = text.strip().lower()
    for w in ("yes", "no"):
        if t.startswith(w):
            return w
    if "yes" in t.split()[:3]: return "yes"
    if "no" in t.split()[:3]: return "no"
    return "no"


def run_short_qa(model, items, store_path, decoding="greedy", qkey="question_id",
                 qtext="text", image_getter=None):
    """Batched single-word answers (POPE, MME). Resumable."""
    store = JsonlStore(store_path, key=qkey)
    todo = [x for x in items if str(x[qkey]) not in store]
    bs = C["batch_short"]; i = 0
    while i < len(todo):
        batch = todo[i:i + bs]
        try:
            imgs = image_getter(batch) if image_getter else load_images([x["path"] for x in batch])
            prompts = [short_prompt(x[qtext]) for x in batch]
            if decoding == "vcd":
                v = C["vcd"]
                outs = vcd_generate_batch(model, imgs, prompts, C["max_new_tokens_short"],
                                          v["alpha"], v["beta"], v["noise_step"], v["seed"] + i)
            else:
                outs = generate_batch(model, imgs, prompts, C["max_new_tokens_short"])
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache(); bs = max(1, bs // 2); print(f"  OOM -> batch {bs}"); continue
        store.add_many([{qkey: x[qkey], "raw": o[0]["text"], "pred": parse_yes_no(o[0]["text"])}
                        for x, o in zip(batch, outs)])
        i += len(batch)
    return store


with ledger("model_download"):
    ensure_models()
PROC = load_processor()
print("models:", MODEL_DIRS)


[ledger] model_download: 0.0 min, peak 0.02 GB (ok)
models: {'vlm': '/content/aavlm_local/models/llava-1.5-7b-hf', 'detector': '/content/aavlm_local/models/grounding-dino-tiny', 'matcher': '/content/aavlm_local/models/all-MiniLM-L6-v2'}


## 6 · Hallucination-Aware Reward and preference pairs
$R(c,I)=\alpha S(c)-\lambda H(c,I)$. $H$ = share of a caption's object entities that Grounding DINO cannot ground when queried with exactly those entities; $S$ = reference unigram recall. All variants (controls, ablations, λ sweep) are derived from one table of per-sample scores, so they cost no extra model passes.

In [ ]:
# ============================================================================
# 6) HALLUCINATION-AWARE REWARD (HAR) and preference-pair construction
# ============================================================================
#   R(c, I) = alpha * S(c, refs) - lambda * H(c, I)
#   H = fraction of the caption's object entities that the open-vocabulary
#       detector cannot ground when it is queried with exactly those entities
#   S = reference unigram recall: max over references of the fraction of a
#       reference's word types that also occur in the caption.
# NOTE (honesty fix): v2 called this term "normalized CIDEr", but the code passed
# MockCider (exactly this unigram recall, x2/2). CIDEr-D is unsuitable here: its
# Gaussian length penalty (sigma = 6 words) drives every paragraph-length caption
# to ~0 against one-sentence COCO references. v3 names the term accurately.
import re
from collections import Counter

COCO80 = ["person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
          "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog",
          "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
          "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite",
          "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle",
          "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
          "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant",
          "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
          "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
          "teddy bear", "hair drier", "toothbrush"]

NON_OBJECT = {
    "image", "picture", "photo", "photograph", "view", "scene", "background", "foreground", "side",
    "area", "part", "front", "back", "top", "bottom", "color", "colour", "way", "type", "kind",
    "number", "amount", "lot", "something", "someone", "thing", "things", "left", "right", "middle",
    "center", "centre", "distance", "day", "time", "moment", "shot", "edge", "row", "piece", "group",
    "pair", "couple", "variety", "assortment", "arrangement", "display", "setting", "surface",
    "object", "item", "detail", "feature", "corner", "section", "portion", "size", "shape",
    "material", "pattern", "design", "style", "appearance", "atmosphere", "mood", "focus", "angle",
    "perspective", "frame", "border", "edges", "scenery", "environment", "surroundings", "location",
    "place", "spot", "position", "direction", "sense", "manner", "fact", "look", "presence",
    "addition"}

_NLP = None


def nlp():
    global _NLP
    if _NLP is None:
        import spacy
        _NLP = spacy.load("en_core_web_sm", disable=["ner"])
    return _NLP


def entities_batch(texts):
    """One noun-only entity per noun chunk (determiners/adjectives stripped)."""
    out = []
    for doc in nlp().pipe(texts, batch_size=256):
        ents = set()
        for ch in doc.noun_chunks:
            if ch.root.pos_ == "PRON":
                continue
            nouns = [t.lemma_.lower() for t in ch if t.pos_ in {"NOUN", "PROPN"}
                     and t.lemma_.lower() not in NON_OBJECT and not t.is_stop]
            e = " ".join(nouns).strip()
            if len(e) > 1:
                ents.add(e)
        out.append(sorted(ents))
    return out


def ref_unigram_recall(caption, refs):
    cw = set(re.findall(r"[a-z]+", caption.lower()))
    best = 0.0
    for r in refs:
        rw = set(re.findall(r"[a-z]+", r.lower()))
        if rw:
            best = max(best, len(cw & rw) / len(rw))
    return best


# ------------------------------------------------------------------ detector
class Detector:
    """Grounding DINO Swin-T (native Transformers port), batched."""

    def __init__(self):
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        import inspect
        self.proc = AutoProcessor.from_pretrained(MODEL_DIRS["detector"])
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(
            MODEL_DIRS["detector"]).to("cuda" if torch.cuda.is_available() else "cpu").eval()
        sig = inspect.signature(self.proc.post_process_grounded_object_detection).parameters
        self.box_kw = "threshold" if "threshold" in sig else "box_threshold"

    @staticmethod
    def prompt(phrases):
        seen, uniq = set(), []
        for p in phrases:
            p = p.strip().lower()
            if p and p not in seen:
                seen.add(p); uniq.append(p)
        return ". ".join(uniq) + "." if uniq else ""

    @torch.no_grad()
    def detect(self, images, phrase_lists):
        d = C["detector"]; res = [None] * len(images)
        idx = [i for i, pl in enumerate(phrase_lists) if self.prompt(pl)]
        for i in range(len(images)):
            if i not in idx:
                res[i] = {"entities": [], "raw": []}
        if not idx:
            return res
        ims = [images[i] for i in idx]; texts = [self.prompt(phrase_lists[i]) for i in idx]
        inp = self.proc(images=ims, text=texts, return_tensors="pt", padding=True,
                        truncation=True, max_length=256).to(self.model.device)
        out = self.model(**inp)
        post = self.proc.post_process_grounded_object_detection(
            out, inp["input_ids"], **{self.box_kw: d["box_threshold"]},
            text_threshold=d["text_threshold"], target_sizes=[im.size[::-1] for im in ims])
        for j, r in zip(idx, post):
            labels = r.get("text_labels", r.get("labels"))
            raw, ents = [], set()
            for s, b, l in zip(r["scores"].tolist(), r["boxes"].tolist(), labels):
                l = str(l).lower().strip()
                raw.append({"label": l, "score": round(s, 4), "box": [round(x, 1) for x in b]})
                if s >= d["keep_score"] and l:
                    ents.add(l)
            res[j] = {"entities": sorted(ents), "raw": raw}
        return res


def run_detection(records, samples_store, store_path, mode, captions=None):
    """mode='cc'     : query = union of entities over the image's own captions
       mode='coco80' : fixed 80-category query (ablation)."""
    store = JsonlStore(store_path, key="image_id")
    todo = [r for r in records if r["image_id"] not in store]
    if not todo:
        return store
    det = Detector(); bs = C["detector"]["batch"]; t0 = time.time()
    for bi, batch in enumerate(chunks(todo, bs)):
        if mode == "coco80":
            queries = [COCO80] * len(batch)
        else:
            texts_per = [captions[str(r["image_id"])] if captions is not None
                         else samples_store[r["image_id"]]["samples"] for r in batch]
            queries = []
            for texts in texts_per:
                cnt = Counter(e for ents in entities_batch(texts) for e in ents)
                queries.append([e for e, _ in sorted(cnt.items(), key=lambda kv: (-kv[1], len(kv[0]), kv[0]))]
                               [:C["detector"]["max_phrases"]])
        outs = det.detect(load_images([r["path"] for r in batch]), queries)
        store.add_many([{"image_id": r["image_id"], "queried": q, **o}
                        for r, q, o in zip(batch, queries, outs)])
        if bi % 50 == 0:
            print(f"  detection[{mode}] {len(store)}/{len(records)} ({(time.time()-t0)/60:.1f} min)")
    free_gpu(det)
    return store


# ------------------------------------------------------------------ matcher
class Matcher:
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.m = SentenceTransformer(MODEL_DIRS["matcher"],
                                     device="cuda" if torch.cuda.is_available() else "cpu")
        self.cache = {}

    def embed(self, strings):
        new = sorted({s for s in strings if s not in self.cache})
        if new:
            E = self.m.encode(new, batch_size=512, normalize_embeddings=True, convert_to_numpy=True)
            self.cache.update(dict(zip(new, E)))

    def hallucinated(self, cap_ents, vis_ents):
        if not cap_ents:
            return []
        if not vis_ents:
            return list(cap_ents)
        A = np.stack([self.cache[e] for e in cap_ents]); B = np.stack([self.cache[v] for v in vis_ents])
        best = (A @ B.T).max(axis=1)
        return [e for e, s in zip(cap_ents, best) if s < C["har"]["tau"]]


def score_samples(records, samples_store, det_stores, store_path):
    """Per-sample reward components for every training image (all variants
    are derived from this one table, so ablations cost no extra scoring)."""
    store = JsonlStore(store_path, key="image_id")
    todo = [r for r in records if r["image_id"] not in store and r["image_id"] in samples_store]
    if not todo:
        return store
    matcher = Matcher()
    for batch in chunks(todo, 500):
        texts = [t for r in batch for t in samples_store[r["image_id"]]["samples"]]
        ents = entities_batch(texts)
        vis_all = [v for r in batch for ds in det_stores.values()
                   for v in ds[r["image_id"]]["entities"]]
        matcher.embed([e for es in ents for e in es] + vis_all)
        recs, k = [], 0
        for r in batch:
            sm = samples_store[r["image_id"]]; rows = []
            for j, t in enumerate(sm["samples"]):
                e = ents[k]; k += 1
                row = {"text": t, "finished": sm["finished"][j], "n_tokens": sm["n_tokens"][j],
                       "n_words": len(t.split()), "entities": e,
                       "S": round(ref_unigram_recall(t, r["refs"]), 4)}
                for name, ds in det_stores.items():
                    h = matcher.hallucinated(e, ds[r["image_id"]]["entities"])
                    row[f"H_{name}"] = round(len(h) / max(len(e), 1), 4)
                    row[f"halluc_{name}"] = h
                rows.append(row)
            recs.append({"image_id": r["image_id"], "rows": rows})
        store.add_many(recs)
        print(f"  scored {len(store)}/{len(records)} images")
    free_gpu(matcher)
    return store


# ------------------------------------------------------------------ pairs
def _cands(rows):
    seen, out = set(), []
    for r in rows:
        t = r["text"].strip()
        if (r["finished"] or not C["require_finished"]) and t and t not in seen:
            seen.add(t); out.append(r)
    return out


def _pair(image_id, path, ch, rj, margin, extra=None):
    return {"image_id": image_id, "image": path, "prompt": caption_prompt(),
            "chosen": ch["text"], "rejected": rj["text"], "margin": round(margin, 4),
            "H_chosen": ch["H_cc"], "H_rejected": rj["H_cc"], "S_chosen": ch["S"], "S_rejected": rj["S"],
            "words_chosen": ch["n_words"], "words_rejected": rj["n_words"], **(extra or {})}


def build_pairs(variant, records, scores, matched_ids=None, seed=0):
    """variant: dict(kind='har', alpha, lam, H='cc'|'coco80')   reward-ranked
                dict(kind='length')   chosen = shortest, rejected = longest (brevity control)
                dict(kind='random')   two random samples, random orientation (no-signal control)"""
    path = {r["image_id"]: r["path"] for r in records}
    rng = random.Random(seed); pairs = []; skipped = 0
    ids = matched_ids if matched_ids is not None else [r["image_id"] for r in records]
    for iid in ids:
        if iid not in scores:
            skipped += 1; continue
        c = _cands(scores[iid]["rows"])
        if len(c) < 2:
            skipped += 1; continue
        if variant["kind"] == "har":
            hk = f"H_{variant.get('H', 'cc')}"
            rew = [variant["alpha"] * x["S"] - variant["lam"] * x[hk] for x in c]
            order = sorted(range(len(c)), key=lambda i: -rew[i])   # stable
            b, w = order[0], order[-1]
            m = rew[b] - rew[w]
            if m < C["har"]["min_margin"]:
                skipped += 1; continue
            pairs.append(_pair(iid, path[iid], c[b], c[w], m,
                               {"r_chosen": round(rew[b], 4), "r_rejected": round(rew[w], 4)}))
        elif variant["kind"] == "length":
            order = sorted(range(len(c)), key=lambda i: c[i]["n_words"])
            b, w = order[0], order[-1]
            if c[b]["n_words"] == c[w]["n_words"]:
                skipped += 1; continue
            pairs.append(_pair(iid, path[iid], c[b], c[w], c[w]["n_words"] - c[b]["n_words"]))
        elif variant["kind"] == "random":
            b, w = rng.sample(range(len(c)), 2)
            pairs.append(_pair(iid, path[iid], c[b], c[w], 0.0))
        else:
            raise ValueError(variant)
    return pairs, skipped


def pair_stats(pairs, skipped):
    if not pairs:
        return {"n_pairs": 0, "skipped": skipped}
    a = lambda k: float(np.mean([p[k] for p in pairs]))
    return {"n_pairs": len(pairs), "skipped": skipped,
            "mean_margin": round(a("margin"), 4),
            "H_chosen": round(a("H_chosen"), 4), "H_rejected": round(a("H_rejected"), 4),
            "S_chosen": round(a("S_chosen"), 4), "S_rejected": round(a("S_rejected"), 4),
            "words_chosen": round(a("words_chosen"), 1), "words_rejected": round(a("words_rejected"), 1),
            # brevity diagnostic: how often the preferred caption is the shorter one
            "frac_chosen_shorter": round(float(np.mean([p["words_chosen"] < p["words_rejected"]
                                                        for p in pairs])), 4)}


## 7 · HAR-DPO training
LoRA (language model only) + TRL `DPOTrainer`; reference policy = adapter disabled. Each run saves its exact training pairs, log, metadata and measured runtime.

In [ ]:
# ============================================================================
# 7) HAR-DPO TRAINING — LoRA + DPO (offline form of KL-regularised RL)
# ============================================================================
# DPO optimises the same objective as RLHF's KL-regularised reward
# maximisation, max_pi E[r] - beta*KL(pi || pi_ref), in closed form (Rafailov
# et al., 2023). The reference policy is the frozen starting model, obtained by
# disabling the LoRA adapter (ref_model=None), so only one 7B copy is resident.
#
# Fixes vs v2: (i) LoRA now targets ONLY the language model (v2's
# target_modules also matched q/k/v in the CLIP vision tower: 2.36 M of the
# 42.3 M trainable parameters); (ii) every run uses its own fresh model, so no
# other policy is ever contaminated; (iii) seeds control LoRA init + data order.
import inspect, glob, shutil

LORA_TARGET = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
LORA_TARGET_ALL = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def _filter_kwargs(fn, kw, label):
    params = inspect.signature(fn).parameters
    if any(p.kind == p.VAR_KEYWORD for p in params.values()):
        return kw
    keep = {k: v for k, v in kw.items() if k in params}
    dropped = sorted(set(kw) - set(keep))
    if dropped:
        print(f"[compat] {label} ignores: {dropped}")
    return keep


def pairs_dataset(pairs):
    from datasets import Dataset, Features, Sequence, Value, Image as HFImage
    rows = {"images": [[p["image"]] for p in pairs], "prompt": [p["prompt"] for p in pairs],
            "chosen": [p["chosen"] for p in pairs], "rejected": [p["rejected"] for p in pairs]}
    feats = Features({"images": Sequence(HFImage()), "prompt": Value("string"),
                      "chosen": Value("string"), "rejected": Value("string")})
    return Dataset.from_dict(rows, features=feats)


def latest_checkpoint(d):
    cks = sorted(glob.glob(str(Path(d) / "checkpoint-*")), key=lambda x: int(x.rsplit("-", 1)[1]))
    return cks[-1] if cks else None


def train_dpo(run_name, pairs, seed, init_adapters=(), train_cfg=None):
    """Train one LoRA-DPO policy. Resumable from its latest checkpoint."""
    from transformers import set_seed
    from peft import LoraConfig, get_peft_model
    from trl import DPOConfig, DPOTrainer
    t = dict(C["train"]); t.update(train_cfg or {})
    out = P["runs"] / run_name
    final = out / "final_adapter"
    if (final / "adapter_config.json").exists():
        print(f"[train] {run_name}: already trained -> {final}")
        return str(final)
    out.mkdir(parents=True, exist_ok=True)
    jdump(pairs, out / "train_pairs.json")             # exact training data, for provenance
    set_seed(seed)
    model = load_policy({"name": run_name, "adapters": list(init_adapters)})   # fresh copy
    model.config.use_cache = False
    lcfg = LoraConfig(r=t["lora_r"], lora_alpha=t["lora_alpha"], lora_dropout=t["lora_dropout"],
                      bias="none", task_type="CAUSAL_LM",
                      target_modules=LORA_TARGET_ALL if t["lora_on_vision"] else LORA_TARGET)
    model = get_peft_model(model, lcfg)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    vision_lora = [n for n, p in model.named_parameters()
                   if p.requires_grad and "language_model" not in n]
    assert t["lora_on_vision"] or not vision_lora, f"LoRA leaked outside the LM: {vision_lora[:3]}"
    print(f"[train] {run_name}: trainable {n_train:,} / {n_all:,} ({100*n_train/n_all:.3f}%)")

    ds = pairs_dataset(pairs)
    args = dict(output_dir=str(out), per_device_train_batch_size=t["per_device_bs"],
                gradient_accumulation_steps=t["grad_accum"], num_train_epochs=t["epochs"],
                learning_rate=t["lr"], beta=t["beta"], max_length=t["max_length"],
                lr_scheduler_type="linear", warmup_ratio=t["warmup_ratio"],
                logging_steps=t["logging_steps"], save_steps=t["save_steps"], save_total_limit=2,
                bf16=False, fp16=False,      # weights already bf16; AMP scaler not needed (v2-proven)
                gradient_checkpointing=False, remove_unused_columns=False, seed=seed, data_seed=seed,
                report_to=[], dataloader_num_workers=2)
    cfg = DPOConfig(**_filter_kwargs(DPOConfig.__init__, args, "DPOConfig"))
    tkw = dict(model=model, ref_model=None, args=cfg, train_dataset=ds)
    tparams = inspect.signature(DPOTrainer.__init__).parameters
    tkw["processing_class" if "processing_class" in tparams else "tokenizer"] = PROC
    trainer = DPOTrainer(**tkw)
    ck = latest_checkpoint(out)
    if ck:
        print(f"[train] resuming from {ck}")
    with ledger(f"train:{run_name}", n_pairs=len(pairs), seed=seed):
        result = trainer.train(resume_from_checkpoint=ck)
    jdump(trainer.state.log_history, out / "training_log.json")
    trainer.save_model(str(final))
    jdump({"run": run_name, "seed": seed, "n_pairs": len(pairs), "trainable_params": n_train,
           "all_params": n_all, "train_cfg": t, "init_adapters": list(init_adapters),
           "metrics": result.metrics, "global_steps": trainer.state.global_step},
          out / "train_meta.json")
    for ckd in glob.glob(str(out / "checkpoint-*")):      # adapters are kept; optimiser states dropped
        shutil.rmtree(ckd, ignore_errors=True)
    free_gpu(trainer, model)
    return str(final)


## 8 · Evaluation: official CHAIR, POPE, MME

In [ ]:
# ============================================================================
# 8) EVALUATION — official CHAIR, POPE, MME
# ============================================================================
# CHAIR: faithful Python-3 port of the official implementation (Rohrbach et
# al., 2018; github.com/LisaAnne/Hallucination, utils/chair.py), following the
# widely used port in OPERA (Huang et al., 2024): official synonym list
# (downloaded from the official repository), NLTK tokenisation + POS-aware
# WordNet lemmatisation, the official double-word and special-case rules, and
# ground truth = instance annotations UNION objects named in the five
# reference captions. CHAIR_i counts object-word occurrences; Recall is the
# share of ground-truth objects mentioned (OPERA's addition).
# This replaces v2's compact synonym map, which inflated absolute CHAIR.
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

SYN_URL = "https://raw.githubusercontent.com/LisaAnne/Hallucination/master/data/synonyms.txt"


class CHAIR:
    def __init__(self, synonyms_txt, val_ann):
        synonyms = [s.strip().split(", ") for s in synonyms_txt.splitlines() if s.strip()]
        self.mscoco_objects, self.inverse_synonym_dict = [], {}
        for syn in synonyms:
            self.mscoco_objects.extend(syn)
            for s in syn:
                self.inverse_synonym_dict[s] = syn[0]
        self.obj_set = set(self.mscoco_objects)
        coco_double_words = ["motor bike", "motor cycle", "air plane", "traffic light", "street light",
                             "traffic signal", "stop light", "fire hydrant", "stop sign", "parking meter",
                             "suit case", "sports ball", "baseball bat", "baseball glove", "tennis racket",
                             "wine glass", "hot dog", "cell phone", "mobile phone", "teddy bear",
                             "hair drier", "potted plant", "bow tie", "laptop computer", "stove top oven",
                             "hot dog", "teddy bear", "home plate", "train track"]
        animal_words = ["bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra",
                        "giraffe", "animal", "cub"]
        vehicle_words = ["jet", "train"]
        self.double_word_dict = {w: w for w in coco_double_words}
        for a in animal_words:
            self.double_word_dict[f"baby {a}"] = a; self.double_word_dict[f"adult {a}"] = a
        for v in vehicle_words:
            self.double_word_dict[f"passenger {v}"] = v
        self.double_word_dict["bow tie"] = "tie"
        self.double_word_dict["toilet seat"] = "toilet"
        self.double_word_dict["wine glas"] = "wine glass"
        self.val_ann = val_ann
        self.wnl = WordNetLemmatizer()
        self._gt = {}

    @staticmethod
    def _wn_pos(tag):
        return {"J": wordnet.ADJ, "V": wordnet.VERB, "N": wordnet.NOUN, "R": wordnet.ADV}.get(tag[:1])

    def caption_to_words(self, caption):
        words = nltk.word_tokenize(caption.lower())
        words = [self.wnl.lemmatize(w, pos=self._wn_pos(t) or wordnet.NOUN) for w, t in nltk.pos_tag(words)]
        i, double_words, idxs = 0, [], []
        while i < len(words):
            idxs.append(i)
            dw = " ".join(words[i:i + 2])
            if dw in self.double_word_dict:
                double_words.append(self.double_word_dict[dw]); i += 2
            else:
                double_words.append(words[i]); i += 1
        words = double_words
        if ("toilet" in words) & ("seat" in words):
            words = [w for w in words if w != "seat"]
        idxs = [idxs[k] for k, w in enumerate(words) if w in self.obj_set]
        words = [w for w in words if w in self.obj_set]
        node_words = [self.inverse_synonym_dict[w] for w in words]
        return words, node_words, idxs, double_words

    def gt_objects(self, image_id):
        k = str(image_id)
        if k not in self._gt:
            a = self.val_ann.get(k, {"cats": [], "caps": []})
            objs = [self.inverse_synonym_dict[c] for c in a["cats"]]
            for cap in a["caps"]:
                objs.extend(self.caption_to_words(cap)[1])
            self._gt[k] = set(objs)
        return self._gt[k]

    def evaluate(self, captions):
        """captions: {image_id: text} -> (overall, per_image)"""
        per, n_caps, n_hcaps, n_hw, n_cw, n_len, n_rec, n_gt = {}, 0, 0, 0, 0, 0, 0, 0
        for iid, cap in captions.items():
            words, nodes, idxs, raw = self.caption_to_words(cap)
            gt = self.gt_objects(iid)
            hall = [(w, nw) for w, nw in zip(words, nodes) if nw not in gt]
            rec = {nw for nw in nodes if nw in gt}
            per[str(iid)] = {"mentioned": nodes, "hallucinated": [h[1] for h in hall],
                             "hallucinated_words": [h[0] for h in hall], "gt": sorted(gt),
                             "n_words": len(raw), "recall": len(rec) / len(gt) if gt else 0.0,
                             "n_recall": len(rec), "n_gt": len(gt), "chair_s": int(bool(hall)),
                             "chair_i": len(hall) / len(words) if words else 0.0}
            n_caps += 1; n_hcaps += int(bool(hall)); n_hw += len(hall); n_cw += len(nodes)
            n_len += len(raw); n_rec += len(rec); n_gt += len(gt)
        overall = {"CHAIRs": 100 * n_hcaps / max(n_caps, 1), "CHAIRi": 100 * n_hw / max(n_cw, 1),
                   "Recall": 100 * n_rec / max(n_gt, 1), "avg_words": n_len / max(n_caps, 1),
                   "n_captions": n_caps, "n_object_mentions": n_cw, "n_hallucinated_mentions": n_hw}
        return overall, per


def load_chair(val_ann):
    f = P["cache"] / "chair_synonyms.txt"
    if not f.exists():
        download(SYN_URL, f)
    return CHAIR(f.read_text(), val_ann)


# ------------------------------------------------------------------ POPE / MME
def score_yes_no(preds, labels):
    tp = sum(p == "yes" and l == "yes" for p, l in zip(preds, labels))
    tn = sum(p == "no" and l == "no" for p, l in zip(preds, labels))
    fp = sum(p == "yes" and l == "no" for p, l in zip(preds, labels))
    fn = sum(p == "no" and l == "yes" for p, l in zip(preds, labels))
    n = max(len(preds), 1); prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return {"Accuracy": 100 * (tp + tn) / n, "Precision": 100 * prec, "Recall": 100 * rec,
            "F1": 100 * 2 * prec * rec / max(prec + rec, 1e-9),
            "Yes_ratio": 100 * sum(p == "yes" for p in preds) / n, "n": len(preds)}


MME_PERCEPTION = ["existence", "count", "position", "color", "posters", "celebrity", "scene",
                  "landmark", "artwork", "OCR"]
MME_COGNITION = ["commonsense_reasoning", "numerical_calculation", "text_translation", "code_reasoning"]
MME_HALLUCINATION = ["existence", "count", "position", "color"]


def score_mme(rows):
    """rows: [{category, question_id, pred, label}] -> per-category acc + acc+ (MME protocol)."""
    by_cat = {}
    for r in rows:
        by_cat.setdefault(r["category"], []).append(r)
    out = {}
    for cat, rs in by_cat.items():
        acc = 100 * np.mean([r["pred"] == r["label"] for r in rs])
        groups = {}
        for r in rs:
            groups.setdefault(r["question_id"], []).append(r["pred"] == r["label"])
        acc_plus = 100 * np.mean([all(v) for v in groups.values()])
        out[cat] = {"acc": acc, "acc_plus": acc_plus, "score": acc + acc_plus, "n": len(rs)}
    agg = lambda cats: sum(out[c]["score"] for c in cats if c in out)
    out["_total"] = {"perception": agg(MME_PERCEPTION), "cognition": agg(MME_COGNITION),
                     "hallucination_subset": agg(MME_HALLUCINATION)}
    return out


# ------------------------------------------------------------------ policy evaluation
def evaluate_policy(name, spec, tasks, decoding="greedy"):
    """Run the requested tasks for one policy; every output is cached on Drive.
    Returns a dict of metrics (also written to eval/<name>/metrics.json)."""
    d = P["eval"] / name; d.mkdir(parents=True, exist_ok=True)
    mpath = d / "metrics.json"; metrics = jload(mpath, {})
    need = [t for t in tasks if t not in metrics]
    if not need:
        return metrics
    model = load_policy(spec)
    try:
        if "chair" in need:
            with ledger(f"eval:{name}:chair"):
                caps = run_captions(model, D["chair"], d / "chair_captions.jsonl", decoding=decoding, desc=name)
            overall, per = CHAIR_EVAL.evaluate({k: v["caption"] for k, v in caps.items()})
            overall["avg_tokens"] = float(np.mean([v["n_tokens"] for v in caps.values()]))
            overall["frac_finished"] = float(np.mean([v["finished"] for v in caps.values()]))
            jdump(per, d / "chair_per_image.json"); metrics["chair"] = overall
        if "pope" in need:
            res = {}
            for s, items in D["pope"].items():
                with ledger(f"eval:{name}:pope_{s}"):
                    st = run_short_qa(model, items, d / f"pope_{s}.jsonl", decoding=decoding)
                preds = [st[x["question_id"]]["pred"] for x in items]
                res[s] = score_yes_no(preds, [x["label"] for x in items])
            metrics["pope"] = res
        if "mme" in need and MME is not None:
            items = [{"qid": i, "question_id": r["question_id"], "category": r["category"],
                      "text": r["question"], "label": r["answer"].strip().lower()}
                     for i, r in enumerate(MME.remove_columns("image"))]
            getter = lambda b: [MME[x["qid"]]["image"].convert("RGB") for x in b]
            with ledger(f"eval:{name}:mme"):
                st = run_short_qa(model, items, d / "mme.jsonl", decoding=decoding, qkey="qid",
                                  image_getter=getter)
            metrics["mme"] = score_mme([{**x, "pred": st[x["qid"]]["pred"]} for x in items])
    finally:
        free_gpu(model)
    jdump(metrics, mpath)
    return metrics


CHAIR_EVAL = load_chair(D["val_ann"])
print("CHAIR evaluator ready:", len(CHAIR_EVAL.inverse_synonym_dict), "surface forms")


CHAIR evaluator ready: 403 surface forms


## 9 · Statistics

In [ ]:
# ============================================================================
# 9) STATISTICS — seed aggregation, paired tests, bootstrap CIs, reward validity
# ============================================================================
# Two distinct sources of uncertainty are reported separately:
#   * training-seed variance: mean ± SD over independent DPO runs (seeds)
#   * evaluation-sample variance: paired bootstrap over evaluation images and
#     exact paired tests (Wilcoxon signed-rank; McNemar), Holm-corrected.
from scipy import stats as sps


def mean_sd(xs):
    xs = [float(x) for x in xs if x is not None]
    if not xs:
        return None, None, 0
    return float(np.mean(xs)), (float(np.std(xs, ddof=1)) if len(xs) > 1 else 0.0), len(xs)


def chair_arrays(per, ids):
    a = lambda k: np.array([per[str(i)][k] for i in ids], dtype=float)
    return {"h": np.array([len(per[str(i)]["hallucinated"]) for i in ids], float),
            "m": np.array([len(per[str(i)]["mentioned"]) for i in ids], float),
            "s": a("chair_s"), "r": a("n_recall"), "g": a("n_gt"), "w": a("n_words")}


def _chair_stats(x, idx):
    h, m, s, r, g, w = (x[k][idx] for k in ("h", "m", "s", "r", "g", "w"))
    return np.array([100 * s.mean(), 100 * h.sum() / max(m.sum(), 1), 100 * r.sum() / max(g.sum(), 1),
                     w.mean()])


CHAIR_KEYS = ["CHAIRs", "CHAIRi", "Recall", "avg_words"]


def paired_chair_test(per_base, per_ours, B=None, seed=0):
    """Paired image-level comparison of two policies on the SAME images."""
    B = B or C["n_bootstrap"]
    ids = sorted(set(per_base) & set(per_ours), key=int)
    xb, xo = chair_arrays(per_base, ids), chair_arrays(per_ours, ids)
    n = len(ids); rng = np.random.default_rng(seed)
    point = _chair_stats(xo, np.arange(n)) - _chair_stats(xb, np.arange(n))
    boot = np.empty((B, 4))
    for b in range(B):
        idx = rng.integers(0, n, n)
        boot[b] = _chair_stats(xo, idx) - _chair_stats(xb, idx)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    d = xo["h"] - xb["h"]                     # per-image hallucinated-mention count change
    wil = sps.wilcoxon(d, zero_method="wilcox").pvalue if np.any(d != 0) else 1.0
    bc = int(np.sum((xb["s"] == 1) & (xo["s"] == 0))); cb = int(np.sum((xb["s"] == 0) & (xo["s"] == 1)))
    mcn = sps.binomtest(min(bc, cb), bc + cb, 0.5).pvalue if bc + cb else 1.0
    return {"n_images": n,
            "delta": dict(zip(CHAIR_KEYS, point.round(3).tolist())),
            "ci95": {k: [round(float(lo[i]), 3), round(float(hi[i]), 3)] for i, k in enumerate(CHAIR_KEYS)},
            "wilcoxon_p_halluc_count": float(wil),
            "mcnemar_chairs": {"base_only_halluc": bc, "ours_only_halluc": cb, "p": float(mcn)},
            "boot_dCHAIRi": boot[:, 1].round(3).tolist(), "boot_dCHAIRs": boot[:, 0].round(3).tolist(),
            "images_improved": int(np.sum(d < 0)), "images_worsened": int(np.sum(d > 0)),
            "images_unchanged": int(np.sum(d == 0))}


def paired_pope_test(store_base, store_ours, items):
    cb = np.array([store_base[x["question_id"]]["pred"] == x["label"] for x in items])
    co = np.array([store_ours[x["question_id"]]["pred"] == x["label"] for x in items])
    b = int(np.sum(~cb & co)); c = int(np.sum(cb & ~co))
    p = sps.binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    return {"ours_fixes": b, "ours_breaks": c, "p": float(p),
            "delta_acc": round(100 * (co.mean() - cb.mean()), 3)}


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (dict name -> p)."""
    items = sorted(pvals.items(), key=lambda kv: kv[1]); m = len(items); out, running = {}, 0.0
    for i, (k, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p)); out[k] = running
    return out


def reward_validity(per_chair, det_store, captions):
    """Does the annotation-free HAR hallucination rate agree with annotation-based
    CHAIR on held-out captions?  Spearman correlation and AUC for detecting
    captions that CHAIR flags as hallucinated."""
    m = Matcher(); ids = [i for i in per_chair if i in captions and int(i) in det_store]
    ents = entities_batch([captions[i] for i in ids])
    m.embed([e for es in ents for e in es] + [v for i in ids for v in det_store[int(i)]["entities"]])
    H = np.array([len(m.hallucinated(e, det_store[int(i)]["entities"])) / max(len(e), 1)
                  for i, e in zip(ids, ents)])
    ci = np.array([per_chair[i]["chair_i"] for i in ids]); cs = np.array([per_chair[i]["chair_s"] for i in ids])
    free_gpu(m)
    rho = sps.spearmanr(H, ci)
    auc = None
    if 0 < cs.sum() < len(cs):
        u = sps.mannwhitneyu(H[cs == 1], H[cs == 0], alternative="two-sided").statistic
        auc = float(u / (cs.sum() * (len(cs) - cs.sum())))
    return {"n": len(ids), "spearman_rho": float(rho.statistic), "spearman_p": float(rho.pvalue),
            "auc_chairs": auc, "H": H.round(4).tolist(), "chair_i": ci.round(4).tolist(),
            "chair_s": cs.astype(int).tolist()}


## 10 · Experiment registry and stages

In [ ]:
# ============================================================================
# 10) EXPERIMENT REGISTRY & STAGES
# ============================================================================
# Every trained policy uses the SAME base model, the SAME on-policy samples and
# the SAME training schedule; only the preference signal changes. Controls are
# "matched": they use exactly the images retained by the main HAR variant, so
# dataset size cannot explain a difference.
#
#  Tier 1 — needed for the main claims
#    har_main      HAR-DPO (caption-conditioned grounding, alpha=1, lambda=2), all seeds
#    ctrl_length   brevity control: prefer the SHORTEST sample (tests "it just says less")
#    abl_coco80    grounding ablation: fixed COCO-80 detector query
#    abl_h_only    quality-term ablation: alpha = 0 (penalty only)
#    base+VCD      training-free decoding baseline (Leng et al., 2024)
#    ours+VCD      complementarity of training and decoding
#  Tier 2 — sensitivity / RL iteration
#    ctrl_random   no-signal control: random pairs
#    lam1, lam4    penalty-weight sensitivity (lambda = 1, 4)
#    iter2         second on-policy round: re-sample from the round-1 policy on
#                  new images, re-score, train a new adapter on top (policy iteration)
S0 = C["seeds"][0]
HAR = dict(kind="har", alpha=C["har"]["alpha"], lam=C["har"]["lam"], H="cc")

EXPERIMENTS = [
    dict(name="har_main", label="Auto Align-VLM (HAR-DPO)", tier=1, variant=HAR,
         seeds=C["seeds"], tasks=["chair", "pope", "mme"]),
    dict(name="ctrl_length", label="Length-preference control", tier=1,
         variant=dict(kind="length"), matched="har_main", seeds=[S0], tasks=["chair", "pope"]),
    dict(name="abl_coco80", label="HAR w/ fixed COCO-80 grounding", tier=1,
         variant=dict(HAR, H="coco80"), seeds=[S0], tasks=["chair", "pope"]),
    dict(name="abl_h_only", label="HAR w/o quality term (alpha=0)", tier=1,
         variant=dict(HAR, alpha=0.0), seeds=[S0], tasks=["chair", "pope"]),
    dict(name="ctrl_random", label="Random-preference control", tier=2,
         variant=dict(kind="random"), matched="har_main", seeds=[S0], tasks=["chair", "pope"]),
    dict(name="lam1", label="HAR, lambda = 1", tier=2, variant=dict(HAR, lam=1.0),
         seeds=[S0], tasks=["chair", "pope"]),
    dict(name="lam4", label="HAR, lambda = 4", tier=2, variant=dict(HAR, lam=4.0),
         seeds=[S0], tasks=["chair", "pope"]),
    dict(name="iter2", label="Auto Align-VLM, 2nd on-policy round", tier=2, variant=HAR,
         seeds=[S0], tasks=["chair", "pope", "mme"], round=2, init="har_main"),
]
DECODING_BASELINES = [
    dict(name="base_vcd", label="LLaVA-1.5 + VCD", tier=1, adapters_from=None, tasks=["chair", "pope"]),
    dict(name="ours_vcd", label="Auto Align-VLM + VCD", tier=1, adapters_from="har_main", tasks=["chair", "pope"]),
]
ACTIVE = [e for e in EXPERIMENTS if e["tier"] in RUN_TIERS]
ACTIVE_DEC = [e for e in DECODING_BASELINES if e["tier"] in RUN_TIERS]
LABELS = {"base": "LLaVA-1.5-7B (base)", **{e["name"]: e["label"] for e in EXPERIMENTS + DECODING_BASELINES}}


def run_name(exp, seed):
    return f"{exp['name']}_s{seed}"


def adapter_dir(name):
    return str(P["runs"] / name / "final_adapter")


def policy_spec(exp, seed):
    ad = []
    if exp.get("init"):
        ad.append(adapter_dir(f"{exp['init']}_s{seed}"))
    ad.append(adapter_dir(run_name(exp, seed)))
    return {"name": run_name(exp, seed), "adapters": ad}


# ------------------------------------------------------------------ stages
def stage_base_eval():
    return evaluate_policy("base", {"name": "base", "adapters": []}, ["chair", "pope", "mme"])


def stage_round_data(round_id, records, sampler_spec):
    """On-policy samples -> detections -> per-sample reward components."""
    samples_path = P["cache"] / f"samples_r{round_id}.jsonl"
    st = JsonlStore(samples_path, key="image_id")
    if len(st) < len(records):
        model = load_policy(sampler_spec)
        with ledger(f"sampling_r{round_id}", n_images=len(records), k=C["sampling"]["k"]):
            st = run_sampling(model, records, samples_path, seed_base=C["data_seed"] * 10 + round_id)
        free_gpu(model)
    modes = ["cc"] + (["coco80"] if round_id == 1 and any(
        e["variant"].get("H") == "coco80" for e in ACTIVE) else [])
    dets = {}
    for m in modes:
        with ledger(f"detection_{m}_r{round_id}"):
            dets[m] = run_detection(records, st, P["cache"] / f"detections_{m}_r{round_id}.jsonl", m)
    with ledger(f"scoring_r{round_id}"):
        sc = score_samples(records, st, dets, P["cache"] / f"scores_r{round_id}_{'-'.join(modes)}.jsonl")
    return st, dets, sc


def stage_pairs(exp, records, scores):
    f = P["cache"] / "pairs" / f"{exp['name']}.json"
    cached = jload(f)
    if cached is not None:
        return cached["pairs"], cached["stats"]
    matched = None
    if exp.get("matched"):
        matched = [p["image_id"] for p in jload(P["cache"] / "pairs" / f"{exp['matched']}.json")["pairs"]]
    pairs, skipped = build_pairs(exp["variant"], records, scores, matched_ids=matched, seed=C["data_seed"])
    stats = pair_stats(pairs, skipped)
    jdump({"variant": exp["variant"], "stats": stats, "pairs": pairs}, f)
    print(f"[pairs] {exp['name']}: {stats}")
    return pairs, stats


def stage_train_and_eval(exp, pairs):
    need = C["train"]["per_device_bs"] * C["train"]["grad_accum"]
    if len(pairs) < need:
        print(f"[skip] {exp['name']}: only {len(pairs)} pairs (< one optimisation step of {need}).")
        return
    for s in exp["seeds"]:
        init = [adapter_dir(f"{exp['init']}_s{s}")] if exp.get("init") else []
        train_dpo(run_name(exp, s), pairs, seed=s, init_adapters=init)
        evaluate_policy(run_name(exp, s), policy_spec(exp, s), exp["tasks"])


def stage_decoding_baselines():
    for d in ACTIVE_DEC:
        if d["adapters_from"]:
            exp = next(e for e in EXPERIMENTS if e["name"] == d["adapters_from"])
            spec = policy_spec(exp, S0)
        else:
            spec = {"name": d["name"], "adapters": []}
        evaluate_policy(d["name"], spec, d["tasks"], decoding="vcd")


def stage_reward_validity():
    """Detector-grounded H on the base model's held-out CHAIR captions vs CHAIR."""
    out = P["pubdata"] / "reward_validity.json"
    if out.exists():
        return jload(out)
    caps = {k: v["caption"] for k, v in JsonlStore(P["eval"] / "base" / "chair_captions.jsonl",
                                                   key="image_id")._d.items()}
    with ledger("reward_validity_detection"):
        det = run_detection(D["chair"], None, P["cache"] / "detections_cc_eval_base.jsonl", "cc",
                            captions={k: [v] for k, v in caps.items()})
    rv = reward_validity(jload(P["eval"] / "base" / "chair_per_image.json"), det, caps)
    jdump(rv, out)
    return rv


def progress_report():
    """Measured (not guessed) compute so far, and what is still pending."""
    led = read_ledger()
    tr = [r["seconds"] for r in led if r["stage"].startswith("train:") and r["status"] == "ok"]
    done_runs = [e["name"] + f"_s{s}" for e in ACTIVE for s in e["seeds"]
                 if (Path(adapter_dir(run_name(e, s))) / "adapter_config.json").exists()]
    todo_runs = [e["name"] + f"_s{s}" for e in ACTIVE for s in e["seeds"]
                 if run_name(e, s) not in done_runs]
    print(f"GPU hours logged so far: {sum(r['seconds'] for r in led)/3600:.2f}")
    if tr:
        print(f"measured mean DPO run: {np.mean(tr)/3600:.2f} h  ->  remaining training ~"
              f"{np.mean(tr)*len(todo_runs)/3600:.1f} h for {len(todo_runs)} runs")
    print("trained:", done_runs); print("pending:", todo_runs)


## 11 · Results assembly (`results_master.json`)

In [ ]:
# ============================================================================
# 11) RESULTS ASSEMBLY — one machine-readable source of truth for the paper
# ============================================================================
# results_master.json holds every number that appears in a table, figure or
# sentence of the manuscript, with provenance. Nothing is typed by hand.


def _policy_rows():
    rows = [("base", "base", None)]
    for e in ACTIVE:
        rows += [(run_name(e, s), e["name"], s) for s in e["seeds"]]
    rows += [(d["name"], d["name"], None) for d in ACTIVE_DEC]
    return [r for r in rows if (P["eval"] / r[0] / "metrics.json").exists()]


def _flat(m):
    """Flatten a policy's metrics into scalar columns."""
    out = {}
    if "chair" in m:
        for k in ("CHAIRs", "CHAIRi", "Recall", "avg_words", "avg_tokens"):
            out[k] = m["chair"].get(k)
    for s, v in m.get("pope", {}).items():
        for k in ("Accuracy", "Precision", "Recall", "F1", "Yes_ratio"):
            out[f"POPE_{s}_{k}"] = v[k]
    if m.get("pope"):
        out["POPE_avg_F1"] = float(np.mean([v["F1"] for v in m["pope"].values()]))
        out["POPE_avg_Accuracy"] = float(np.mean([v["Accuracy"] for v in m["pope"].values()]))
    if "mme" in m:
        t = m["mme"]["_total"]
        out.update(MME_perception=t["perception"], MME_cognition=t["cognition"],
                   MME_hallucination=t["hallucination_subset"])
        for c in MME_PERCEPTION + MME_COGNITION:
            if c in m["mme"]:
                out[f"MME_{c}"] = m["mme"][c]["score"]
    return out


def category_counts(per):
    cnt = Counter()
    for v in per.values():
        cnt.update(v["hallucinated"])
    return cnt


def collect_results():
    R = {"config": C, "env": ENV, "labels": LABELS, "policies": {}, "groups": {}, "pairs": {},
         "train": {}, "tests": {}, "pope_tests": {}}
    rows = _policy_rows()
    for name, group, seed in rows:
        R["policies"][name] = {"group": group, "seed": seed, "metrics": _flat(jload(P["eval"] / name / "metrics.json")),
                               "mme_detail": jload(P["eval"] / name / "metrics.json").get("mme")}
    # seed aggregation per group
    groups = {}
    for name, v in R["policies"].items():
        groups.setdefault(v["group"], []).append(v["metrics"])
    for gname, ms in groups.items():
        keys = sorted({k for m in ms for k in m})
        R["groups"][gname] = {k: dict(zip(("mean", "sd", "n"), mean_sd([m.get(k) for m in ms]))) for k in keys}
    # preference data + training
    for f in sorted((P["cache"] / "pairs").glob("*.json")):
        R["pairs"][f.stem] = jload(f)["stats"]
    for d in sorted(P["runs"].glob("*/train_meta.json")):
        meta = jload(d); log = jload(d.parent / "training_log.json", [])
        steps = [e for e in log if "loss" in e]
        meta["final_train_loss"] = meta.get("metrics", {}).get("train_loss")
        meta["first_logged_loss"] = steps[0]["loss"] if steps else None
        meta["last_logged_loss"] = steps[-1]["loss"] if steps else None
        R["train"][meta["run"]] = meta
    # paired tests vs base (CHAIR + POPE), Holm-corrected across all comparisons
    base_per = jload(P["eval"] / "base" / "chair_per_image.json")
    if base_per:
        pv = {}
        boot = {}
        for name in R["policies"]:
            if name == "base":
                continue
            per = jload(P["eval"] / name / "chair_per_image.json")
            if per:
                t = paired_chair_test(base_per, per)
                boot[name] = {"dCHAIRi": t.pop("boot_dCHAIRi"), "dCHAIRs": t.pop("boot_dCHAIRs")}
                R["tests"][name] = t; pv[name] = t["wilcoxon_p_halluc_count"]
            pope_t = {}
            for s, items in D["pope"].items():
                fb, fo = P["eval"] / "base" / f"pope_{s}.jsonl", P["eval"] / name / f"pope_{s}.jsonl"
                if fb.exists() and fo.exists():
                    pope_t[s] = paired_pope_test(JsonlStore(fb, "question_id"), JsonlStore(fo, "question_id"), items)
            if pope_t:
                R["pope_tests"][name] = pope_t
        for k, p in holm(pv).items():
            R["tests"][k]["wilcoxon_p_holm"] = p
        jdump(boot, P["pubdata"] / "bootstrap_distributions.json")
        # per-category hallucinated mentions (base vs mean over main seeds)
        cats = {"base": category_counts(base_per)}
        for name, v in R["policies"].items():
            per = jload(P["eval"] / name / "chair_per_image.json")
            if per and name != "base":
                cats[name] = category_counts(per)
        R["categories"] = {k: dict(v) for k, v in cats.items()}
    R["reward_validity"] = {k: v for k, v in (jload(P["pubdata"] / "reward_validity.json") or {}).items()
                            if k not in ("H", "chair_i", "chair_s")}
    # measured compute
    led = read_ledger(); comp = {}
    for r in led:
        key = r["stage"].split(":")[0] if not r["stage"].startswith("eval") else "evaluation"
        comp.setdefault(key, 0.0); comp[key] += r["seconds"] / 3600
    R["compute_gpu_hours"] = {k: round(v, 3) for k, v in comp.items()}
    R["compute_gpu_hours_total"] = round(sum(comp.values()), 3)
    jdump(R, P["pubdata"] / "results_master.json")
    print(f"results_master.json: {len(R['policies'])} policies, {len(R['tests'])} paired tests, "
          f"{R['compute_gpu_hours_total']} GPU-h logged")
    return R


## 12 · Journal figures
Vector PDF + PNG at Elsevier column widths (90 / 140 / 190 mm), embedded TrueType fonts, a validated colour-blind-safe palette, bars that start at zero, no dual axes. A figure whose data does not exist is skipped — never filled with placeholders.

In [ ]:
# ============================================================================
# 12) JOURNAL FIGURES — vector PDF + high-dpi PNG, Elsevier column widths
# ============================================================================
# Every figure is drawn ONLY from results_master.json and the cached raw
# outputs; a figure whose data does not exist yet is skipped, never faked.
# Widths: single column 90 mm, 1.5 column 140 mm, double column 190 mm.
# Fonts are embedded as TrueType (pdf.fonttype 42), text >= 6.5 pt at print size.
# Palette: validated colour-blind-safe categorical order; identity is always
# also carried by text (axis labels / legends), never by colour alone.
import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.offsetbox import TextArea, HPacker, VPacker, AnnotationBbox

MM = 1 / 25.4
W1, W15, W2 = 90 * MM, 140 * MM, 190 * MM
mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "lines.linewidth": 1.5,
    "axes.grid": True, "grid.color": "#e3e2de", "grid.linewidth": 0.5, "axes.axisbelow": True,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.03,
    "axes.titleweight": "bold", "axes.titlelocation": "left"})
INK, INK2, SURF = "#0b0b0b", "#52514e", "#ffffff"
GRAY = "#8c8c88"
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]   # validated order
ROLE = {"base": GRAY, "har_main": CAT[0], "ours_vcd": CAT[0], "iter2": CAT[5],
        "base_vcd": CAT[3], "ctrl_length": CAT[1], "ctrl_random": CAT[4],
        "abl_coco80": CAT[2], "abl_h_only": CAT[2], "lam1": CAT[2], "lam4": CAT[2]}
ORDER = ["base", "base_vcd", "har_main", "ours_vcd", "iter2", "ctrl_length", "ctrl_random",
         "abl_coco80", "abl_h_only", "lam1", "lam4"]
SHORT = {"base": "LLaVA-1.5-7B", "base_vcd": "LLaVA-1.5-7B + VCD", "har_main": "Auto Align-VLM",
         "ours_vcd": "Auto Align-VLM + VCD", "iter2": "Auto Align-VLM (round 2)",
         "ctrl_length": "Length control", "ctrl_random": "Random control",
         "abl_coco80": "w/ COCO-80 grounding", "abl_h_only": "w/o quality term",
         "lam1": "λ = 1", "lam4": "λ = 4"}
HATCH = {"ours_vcd": "////", "base_vcd": "////"}   # decoding-time variants also differ by texture
FIGS = {}


def save(fig, name, caption_note=""):
    paths = []
    for ext in ("pdf", "png"):
        p = P["fig"] / f"{name}.{ext}"
        fig.savefig(p, dpi=C["fig_dpi"] if ext == "png" else None, facecolor=SURF)
        paths.append(str(p))
    plt.close(fig)
    FIGS[name] = {"files": paths, "note": caption_note}
    print("  figure:", name)
    return paths


def _groups_present(R):
    return [g for g in ORDER if g in R["groups"]]


def _gval(R, g, k):
    v = R["groups"].get(g, {}).get(k)
    return (v["mean"], v["sd"], v["n"]) if v and v["mean"] is not None else (None, None, 0)


# --------------------------------------------------------------------------- 1
def fig_pipeline(R=None):
    fig, ax = plt.subplots(figsize=(W2, 62 * MM)); ax.axis("off"); ax.grid(False)
    ax.set_xlim(0, 19); ax.set_ylim(0, 6.2)

    def box(x, y, w, h, txt, fc, bold=False):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                    fc=fc, ec="#3a3a38", lw=0.7))
        ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=6.8,
                weight="bold" if bold else "normal", color=INK)

    def arr(x1, y1, x2, y2, txt=None, rad=0.0, dy=0.18):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=8,
                                     lw=0.8, color="#3a3a38", connectionstyle=f"arc3,rad={rad}"))
        if txt:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2 + dy, txt, ha="center", fontsize=6, color=INK2)

    blue, green, red, sand = "#dbe8f8", "#d9f1e7", "#f9e1e1", "#f4ecd9"
    box(0.2, 3.3, 2.2, 1.6, "Unlabelled\nimage $I$", sand)
    box(3.1, 3.3, 3.0, 1.6, "Policy $\\pi_\\theta$\nLLaVA-1.5-7B\n+ LoRA", blue, True)
    box(6.8, 3.3, 2.6, 1.6, "$k$ sampled\ncaptions\n$c_1 \\ldots c_k$", blue)
    box(10.1, 4.4, 3.2, 1.3, "Entity parser\n$E(c)$ (noun chunks)", green)
    box(10.1, 2.5, 3.2, 1.5, "Grounding DINO\nqueried with $E(c)$\n→ grounded set $V(I)$", green)
    box(14.0, 3.3, 2.4, 1.6, "Reward\n$R=\\alpha S-\\lambda H$", red, True)
    box(16.9, 3.3, 1.9, 1.6, "Pair\n(best,\nworst)", red)
    box(4.9, 0.3, 9.6, 1.3, "DPO update of the LoRA adapter\n(reference = frozen $\\pi_{\\mathrm{ref}}$; offline KL-regularised RL)",
        blue, True)
    arr(2.4, 4.1, 3.1, 4.1); arr(6.1, 4.1, 6.8, 4.1)
    arr(9.4, 4.4, 10.1, 5.0); arr(9.4, 3.8, 10.1, 3.25)
    arr(11.7, 4.4, 11.7, 4.0)
    arr(13.3, 5.0, 14.0, 4.4); arr(13.3, 3.25, 14.0, 3.8)
    arr(16.4, 4.1, 16.9, 4.1)
    arr(17.85, 3.3, 14.5, 0.95, rad=-0.3)
    arr(4.9, 0.95, 4.4, 3.3, rad=-0.3)
    ax.text(18.2, 1.25, "annotation-free\npreference\ndata", fontsize=6, color=INK2, ha="center")
    ax.text(3.0, 1.55, "on-policy\niteration", fontsize=6, color=INK2, ha="center")
    return save(fig, "fig01_pipeline")


# --------------------------------------------------------------------------- 2
def fig_pairs(R):
    f = P["cache"] / "pairs" / "har_main.json"
    if not f.exists():
        return None
    pairs = jload(f)["pairs"]
    fig, axs = plt.subplots(1, 3, figsize=(W2, 52 * MM), gridspec_kw={"wspace": 0.95, "width_ratios": [1, 1, 0.85]})
    bins = np.linspace(0, 1, 21)
    hc = [p["H_chosen"] for p in pairs]; hr = [p["H_rejected"] for p in pairs]
    axs[0].hist(hr, bins=bins, color=CAT[1], alpha=0.85, label=f"rejected (mean {np.mean(hr):.2f})")
    axs[0].hist(hc, bins=bins, color=CAT[0], alpha=0.85, label=f"chosen (mean {np.mean(hc):.2f})")
    axs[0].set(xlabel="detector-grounded hallucination rate $H$", ylabel="preference pairs",
               title="(a) Hallucination rate")
    axs[0].legend(loc="upper right")
    wc = [p["words_chosen"] for p in pairs]; wr = [p["words_rejected"] for p in pairs]
    b2 = np.linspace(0, max(wc + wr) + 5, 26)
    axs[1].hist(wr, bins=b2, color=CAT[1], alpha=0.85, label=f"rejected (mean {np.mean(wr):.0f})")
    axs[1].hist(wc, bins=b2, color=CAT[0], alpha=0.85, label=f"chosen (mean {np.mean(wc):.0f})")
    axs[1].set(xlabel="caption length (words)", ylabel="preference pairs", title="(b) Caption length")
    axs[1].legend(loc="upper right")
    names = [k for k in ["har_main", "abl_coco80", "abl_h_only", "lam1", "lam4"] if k in R["pairs"]]
    y = np.arange(len(names))
    axs[2].barh(y, [R["pairs"][k]["H_rejected"] - R["pairs"][k]["H_chosen"] for k in names],
                color=[ROLE.get(k, GRAY) for k in names], height=0.6)
    axs[2].set_yticks(y, [SHORT[k] for k in names]); axs[2].invert_yaxis()
    for i, k in enumerate(names):
        v = R["pairs"][k]["H_rejected"] - R["pairs"][k]["H_chosen"]
        axs[2].text(max(v, 0), i, f"  n = {R['pairs'][k]['n_pairs']:,}", va="center", fontsize=6, color=INK)
    axs[2].set_xlim(0, max(0.05, max(R["pairs"][k]["H_rejected"] - R["pairs"][k]["H_chosen"] for k in names) * 1.45))
    axs[2].set(xlabel="mean $H$(rejected) $-$ $H$(chosen)", title="(c) Separation by variant")
    axs[2].grid(axis="y", visible=False)
    return save(fig, "fig02_preference_pairs")


# --------------------------------------------------------------------------- 3
def fig_training(R):
    runs = sorted(k for k in R["train"] if k.startswith("har_main_s"))
    if not runs:
        return None
    keys = [("loss", "DPO loss"), ("rewards/margins", "implicit reward margin"),
            ("rewards/accuracies", "implicit reward accuracy")]
    fig, axs = plt.subplots(1, 3, figsize=(W2, 50 * MM), gridspec_kw={"wspace": 0.45})
    for i, r in enumerate(runs):
        log = [e for e in jload(P["runs"] / r / "training_log.json", []) if "loss" in e]
        for ax, (k, lab) in zip(axs, keys):
            st = [e["step"] for e in log if k in e]; v = [e[k] for e in log if k in e]
            if v:
                ax.plot(st, v, color=CAT[i % len(CAT)], lw=1.0, label=f"seed {R['train'][r]['seed']}")
    for ax, (k, lab), t in zip(axs, keys, "abc"):
        ax.set(xlabel="optimisation step", ylabel=lab, title=f"({t}) {lab[0].upper() + lab[1:]}")
    axs[0].legend()
    return save(fig, "fig03_training_dynamics")


# --------------------------------------------------------------------------- 4
def fig_main_chair(R):
    gs = _groups_present(R)
    gs = [g for g in gs if _gval(R, g, "CHAIRi")[0] is not None]
    if len(gs) < 2:
        return None
    fig, axs = plt.subplots(1, 3, figsize=(W2, (22 + 6.2 * len(gs)) * MM), sharey=True,
                            gridspec_kw={"wspace": 0.12})
    y = np.arange(len(gs))
    for ax, (k, lab) in zip(axs, [("CHAIRs", "CHAIR$_S$ (%) ↓"), ("CHAIRi", "CHAIR$_I$ (%) ↓"),
                                  ("Recall", "Object recall (%) ↑")]):
        m = [_gval(R, g, k)[0] for g in gs]; sd = [_gval(R, g, k)[1] or 0 for g in gs]
        bars = ax.barh(y, m, xerr=sd, height=0.62, color=[ROLE.get(g, GRAY) for g in gs],
                       error_kw=dict(lw=0.7, capsize=1.8, ecolor=INK), edgecolor=SURF, linewidth=0.4)
        for b_, g in zip(bars, gs):
            b_.set_hatch(HATCH.get(g, ""))
        for g_i, g in enumerate(gs):           # individual seeds
            seeds = [v["metrics"].get(k) for v in R["policies"].values() if v["group"] == g]
            if len(seeds) > 1:
                ax.scatter(seeds, [g_i] * len(seeds), s=7, color=INK, zorder=3, lw=0)
            ax.text(m[g_i], g_i, f" {m[g_i]:.1f}", va="center", ha="left", fontsize=6.3, color=INK)
        ax.set_xlabel(lab); ax.grid(axis="y", visible=False)
        ax.set_xlim(0, max(1.0, max(mm + s for mm, s in zip(m, sd)) * 1.18))
    axs[0].set_yticks(y, [SHORT.get(g, g) for g in gs]); axs[0].invert_yaxis()
    axs[0].set_title(f"Object hallucination on {C['n_chair']} COCO val2014 images (bars: mean; dots: seeds)",
                     loc="left")
    return save(fig, "fig04_chair_main")


# --------------------------------------------------------------------------- 5
def fig_tradeoff(R):
    gs = [g for g in _groups_present(R) if _gval(R, g, "CHAIRi")[0] is not None]
    if len(gs) < 2:
        return None
    fig, ax = plt.subplots(figsize=(W1 * 1.15, 72 * MM))
    for j, g in enumerate(gs):
        x, xs, _ = _gval(R, g, "Recall"); yv, ys, _ = _gval(R, g, "CHAIRi")
        ax.errorbar(x, yv, xerr=xs or 0, yerr=ys or 0, fmt="o", ms=5, color=ROLE.get(g, GRAY),
                    mec=SURF, mew=0.8, elinewidth=0.7, capsize=1.5)
        ax.annotate(SHORT.get(g, g), (x, yv), xytext=(5, 4 if j % 2 == 0 else -8),
                    textcoords="offset points", fontsize=6, color=INK2)
    ax.set(xlabel="object recall (%) ↑", ylabel="CHAIR$_I$ (%) ↓",
           title="Hallucination vs. coverage")
    ax.annotate("better", xy=(0.93, 0.08), xytext=(0.72, 0.25), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", lw=0.7, color=INK2), fontsize=6.5, color=INK2)
    return save(fig, "fig05_tradeoff")


# --------------------------------------------------------------------------- 6
def fig_pope(R):
    gs = [g for g in ["base", "base_vcd", "har_main", "ours_vcd", "iter2"]
          if _gval(R, g, "POPE_adversarial_F1")[0] is not None]
    if len(gs) < 2:
        return None
    mets = ["Accuracy", "F1", "Precision", "Recall", "Yes_ratio"]
    sets = [s for s in ["random", "popular", "adversarial"] if s in C["pope_settings"]]
    fig, axs = plt.subplots(1, len(sets), figsize=(W2, 58 * MM), sharey=True, gridspec_kw={"wspace": 0.08})
    w = 0.8 / len(gs); x = np.arange(len(mets))
    for ax, s in zip(np.atleast_1d(axs), sets):
        for i, g in enumerate(gs):
            m = [_gval(R, g, f"POPE_{s}_{k}")[0] for k in mets]
            sd = [_gval(R, g, f"POPE_{s}_{k}")[1] or 0 for k in mets]
            ax.bar(x + (i - (len(gs) - 1) / 2) * w, m, w * 0.92, yerr=sd, color=ROLE.get(g, GRAY),
                   label=SHORT.get(g, g), error_kw=dict(lw=0.6, capsize=1.2), hatch=HATCH.get(g, ""),
                   edgecolor=SURF, linewidth=0.4)
        ax.axhline(50, color=INK2, lw=0.6, ls=":")
        ax.set_xticks(x, ["Acc.", "F1", "Prec.", "Rec.", "Yes %"]); ax.set_title(s.capitalize())
        ax.set_ylim(0, 100); ax.grid(axis="x", visible=False)
    np.atleast_1d(axs)[0].set_ylabel("score (%)")
    np.atleast_1d(axs)[0].legend(ncol=len(gs), loc="lower left", bbox_to_anchor=(0, 1.08))
    return save(fig, "fig06_pope")


# --------------------------------------------------------------------------- 7
def fig_length(R):
    base = jload(P["eval"] / "base" / "chair_per_image.json")
    ours = jload(P["eval"] / f"har_main_s{S0}" / "chair_per_image.json")
    if not (base and ours):
        return None
    lc = jload(P["eval"] / f"ctrl_length_s{S0}" / "chair_per_image.json")
    sets = [("base", base), ("har_main", ours)] + ([("ctrl_length", lc)] if lc else [])
    fig, axs = plt.subplots(1, 2, figsize=(W2 * 0.82, 55 * MM), gridspec_kw={"wspace": 0.3})
    allw = [v["n_words"] for _, per in sets for v in per.values()]
    bins = np.linspace(0, max(allw) + 5, 30)
    for g, per in sets:
        axs[0].hist([v["n_words"] for v in per.values()], bins=bins, histtype="step", lw=1.3,
                    color=ROLE[g], label=f"{SHORT[g]} (median {np.median([v['n_words'] for v in per.values()]):.0f})")
    axs[0].set(xlabel="caption length (words)", ylabel="captions", title="(a) Length distribution")
    axs[0].legend(loc="upper right")
    edges = np.unique(np.quantile(allw, [0, .25, .5, .75, 1.0])); edges[-1] += 1
    nb = len(edges) - 1
    centers = []
    for g, per in sets:
        vals = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            sel = [v for v in per.values() if lo <= v["n_words"] < hi]
            h = sum(len(v["hallucinated"]) for v in sel); m = sum(len(v["mentioned"]) for v in sel)
            vals.append(100 * h / m if m else np.nan)
        axs[1].plot(range(nb), vals, "o-", color=ROLE[g], ms=4, label=SHORT[g])
    axs[1].set_xticks(range(nb), [f"{edges[i]:.0f}–{edges[i+1]-1:.0f}" for i in range(nb)])
    axs[1].set(xlabel="caption-length quartile (words)", ylabel="CHAIR$_I$ within bin (%)",
               title="(b) Hallucination at matched length")
    axs[1].legend()
    return save(fig, "fig07_length_analysis")


# --------------------------------------------------------------------------- 8
def fig_lambda(R):
    pts = [(1.0, "lam1", f"lam1_s{S0}"), (C["har"]["lam"], "har_main", f"har_main_s{S0}"),
           (4.0, "lam4", f"lam4_s{S0}")]
    pts = [(l, g, n) for l, g, n in pts if n in R["policies"]]
    if len(pts) < 3:
        return None
    fig, axs = plt.subplots(1, 3, figsize=(W2, 48 * MM), gridspec_kw={"wspace": 0.38})
    for ax, (k, lab) in zip(axs, [("CHAIRi", "CHAIR$_I$ (%) ↓"), ("Recall", "object recall (%) ↑"),
                                  ("avg_words", "mean caption length (words)")]):
        xs = [l for l, _, _ in pts]; ys = [R["policies"][n]["metrics"][k] for _, _, n in pts]
        ax.plot(xs, ys, "o-", color=CAT[0], ms=4)
        b = R["policies"]["base"]["metrics"][k]
        ax.axhline(b, color=GRAY, lw=0.9, ls="--"); ax.text(xs[0], b, "base ", va="bottom", ha="left", fontsize=6, color=INK2)
        ax.set(xlabel="penalty weight λ", ylabel=lab); ax.set_xticks(xs)
    axs[0].set_title(f"Sensitivity to λ (seed {S0})")
    return save(fig, "fig08_lambda_sensitivity")


# --------------------------------------------------------------------------- 9
def fig_categories(R, top=15):
    cats = R.get("categories", {})
    mains = [k for k in cats if k.startswith("har_main_s")]
    if "base" not in cats or not mains:
        return None
    base = Counter(cats["base"]); top_c = [c for c, _ in base.most_common(top)]
    ours = {c: np.mean([cats[m].get(c, 0) for m in mains]) for c in top_c}
    fig, ax = plt.subplots(figsize=(W1 * 1.15, (18 + 5 * len(top_c)) * MM))
    y = np.arange(len(top_c))
    for i, c in enumerate(top_c):
        ax.plot([base[c], ours[c]], [i, i], color="#c9c8c3", lw=1.6, zorder=1)
    ax.scatter([base[c] for c in top_c], y, color=GRAY, s=16, zorder=2, label=SHORT["base"])
    ax.scatter([ours[c] for c in top_c], y, color=CAT[0], s=16, zorder=3,
               label=f"{SHORT['har_main']} (mean of {len(mains)} seeds)")
    ax.set_yticks(y, top_c); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set(xlabel=f"hallucinated mentions ({C['n_chair']} images)", title="Most-hallucinated categories")
    ax.legend(loc="lower right")
    return save(fig, "fig09_categories")


# --------------------------------------------------------------------------- 10
def fig_mme(R):
    if _gval(R, "har_main", "MME_perception")[0] is None or "MME_perception" not in R["policies"]["base"]["metrics"]:
        return None
    cats = [c for c in MME_PERCEPTION + MME_COGNITION if f"MME_{c}" in R["policies"]["base"]["metrics"]]
    fig, ax = plt.subplots(figsize=(W15, (20 + 5.5 * len(cats)) * MM))
    y = np.arange(len(cats)); h = 0.38
    b = [R["policies"]["base"]["metrics"][f"MME_{c}"] for c in cats]
    o = [_gval(R, "har_main", f"MME_{c}")[0] for c in cats]; osd = [_gval(R, "har_main", f"MME_{c}")[1] or 0 for c in cats]
    ax.barh(y - h / 2, b, h * 0.92, color=GRAY, label=SHORT["base"])
    ax.barh(y + h / 2, o, h * 0.92, xerr=osd, color=CAT[0], label=SHORT["har_main"],
            error_kw=dict(lw=0.6, capsize=1.2))
    ax.set_yticks(y, [c.replace("_", " ") + (" *" if c in MME_HALLUCINATION else "") for c in cats])
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.set(xlabel="MME score (acc + acc+, max 200)", title="Capability preservation on MME (* hallucination subset)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12 - 0.9 / len(cats)), ncol=2)
    return save(fig, "fig10_mme")


# --------------------------------------------------------------------------- 11
def fig_bootstrap(R):
    boot = jload(P["pubdata"] / "bootstrap_distributions.json")
    mains = sorted(k for k in (boot or {}) if k.startswith("har_main_s"))
    if not mains:
        return None
    fig, axs = plt.subplots(1, 2, figsize=(W2 * 0.82, 48 * MM), gridspec_kw={"wspace": 0.28})
    for ax, k, lab in [(axs[0], "dCHAIRs", "ΔCHAIR$_S$ (points)"), (axs[1], "dCHAIRi", "ΔCHAIR$_I$ (points)")]:
        for i, m in enumerate(mains):
            ax.hist(boot[m][k], bins=60, histtype="step", lw=1.1, color=CAT[i % 6],
                    label=f"seed {R['policies'][m]['seed']}")
        ax.axvline(0, color=INK, lw=0.8, ls="--")
        ax.set(xlabel=lab + " vs. base", ylabel="bootstrap replicates")
    axs[0].set_title("(a) Paired bootstrap over images"); axs[1].set_title("(b)")
    axs[0].legend()
    return save(fig, "fig11_bootstrap")


# --------------------------------------------------------------------------- 12
def fig_reward_validity(R):
    rv = jload(P["pubdata"] / "reward_validity.json")
    if not rv or rv.get("n", 0) < 5:
        return None
    H, ci, cs = np.array(rv["H"]), np.array(rv["chair_i"]), np.array(rv["chair_s"])
    fig, axs = plt.subplots(1, 2, figsize=(W2 * 0.82, 52 * MM), gridspec_kw={"wspace": 0.3})
    data = [H[cs == 0], H[cs == 1]]
    bp = axs[0].boxplot(data, widths=0.5, patch_artist=True, medianprops=dict(color=INK, lw=1))
    for patch, col in zip(bp["boxes"], [CAT[0], CAT[1]]):
        patch.set_facecolor(col); patch.set_alpha(0.75); patch.set_linewidth(0.6)
    axs[0].set_xticks([1, 2], [f"CHAIR-clean\n(n={len(data[0])})", f"CHAIR-hallucinated\n(n={len(data[1])})"])
    auc = rv.get("auc_chairs")
    axs[0].set(ylabel="HAR hallucination rate $H$", title="(a) $H$ separates captions"
               + (f" (AUC {auc:.2f})" if auc is not None else ""))
    q = np.quantile(H, np.linspace(0, 1, 6)); q[-1] += 1e-9
    xs, ys, ns = [], [], []
    for lo, hi in zip(q[:-1], q[1:]):
        sel = (H >= lo) & (H < hi)
        if sel.sum():
            xs.append(H[sel].mean()); ys.append(100 * ci[sel].mean()); ns.append(int(sel.sum()))
    axs[1].plot(xs, ys, "o-", color=CAT[0], ms=4)
    axs[1].set(xlabel="HAR hallucination rate $H$ (quintile mean)", ylabel="mean per-caption CHAIR$_I$ (%)",
               title=f"(b) Agreement with CHAIR (ρ = {rv['spearman_rho']:.2f})")
    return save(fig, "fig12_reward_validity")


# --------------------------------------------------------------------------- 13
def _highlight_words(text, bad):
    from nltk.stem import WordNetLemmatizer
    wnl = WordNetLemmatizer(); bad = {b.lower() for b in bad}
    bad_tokens = {t for b in bad for t in b.split()}
    out = []
    for w in text.split():
        c = re.sub(r"[^a-z]", "", w.lower())
        out.append((w, bool(c) and (c in bad_tokens or wnl.lemmatize(c) in bad_tokens)))
    return out


def _draw_caption(ax, header, color, words, max_chars, fs=6.3):
    lines, cur, n = [], [], 0
    for w, flag in words:
        if n + len(w) + 1 > max_chars and cur:
            lines.append(cur); cur, n = [], 0
        cur.append((w, flag)); n += len(w) + 1
    if cur:
        lines.append(cur)
    rows = [HPacker(children=[TextArea(w, textprops=dict(color="#b4232a" if f else INK, fontsize=fs,
                                                         weight="bold" if f else "normal")) for w, f in ln],
                    align="baseline", pad=0, sep=2.4) for ln in lines]
    head = TextArea(header, textprops=dict(color=color, fontsize=fs + 0.6, weight="bold"))
    ax.add_artist(AnnotationBbox(VPacker(children=[head] + rows, align="left", pad=0, sep=1.6),
                                 (0, 1), xycoords="axes fraction", box_alignment=(0, 1), frameon=False))
    return len(lines)


def fig_qualitative(R, n_success=3, n_failure=1):
    bpi = jload(P["eval"] / "base" / "chair_per_image.json")
    opi = jload(P["eval"] / f"har_main_s{S0}" / "chair_per_image.json")
    if not (bpi and opi):
        return None
    bc = JsonlStore(P["eval"] / "base" / "chair_captions.jsonl", "image_id")
    oc = JsonlStore(P["eval"] / f"har_main_s{S0}" / "chair_captions.jsonl", "image_id")
    diff = sorted(((len(bpi[k]["hallucinated"]) - len(opi[k]["hallucinated"]), k) for k in bpi if k in opi),
                  reverse=True)
    succ = [k for d, k in diff if d > 0 and len(opi[k]["hallucinated"]) == 0][:n_success]
    fail = [k for d, k in sorted(diff) if d < 0][:n_failure]
    rows = [(k, "success") for k in succ] + [(k, "failure") for k in fail]
    if not rows:
        return None
    wchars = 44
    est = []
    for k, _ in rows:
        nb = len(" ".join(bc[k]["caption"].split())) // wchars + 2
        no = len(" ".join(oc[k]["caption"].split())) // wchars + 2
        est.append(max(nb, no, 9))
    heights = [e * 2.9 + 4 for e in est]
    fig = plt.figure(figsize=(W2, sum(heights) * MM + 0.15))
    gsp = fig.add_gridspec(len(rows), 3, height_ratios=heights, width_ratios=[0.62, 1, 1],
                           hspace=0.12, wspace=0.05)
    for r, (k, kind) in enumerate(rows):
        axi = fig.add_subplot(gsp[r, 0]); axi.grid(False)
        img = np.asarray(Image.open(image_path(int(k), "val2014")).convert("RGB"))
        axi.imshow(img)
        h, w = img.shape[:2]
        axi.set_xlim(-0.5, w - 0.5); axi.set_ylim(h - 0.5, -0.5)   # pin view to the full image
        axi.set_xticks([]); axi.set_yticks([])
        axi.set_title(f"COCO {k}" + ("  (failure case)" if kind == "failure" else ""), fontsize=6.3,
                      loc="left", weight="normal", color=INK2)
        for c, (lab, per, store, col) in enumerate([(SHORT["base"], bpi, bc, INK2),
                                                    (SHORT["har_main"], opi, oc, CAT[0])]):
            ax = fig.add_subplot(gsp[r, c + 1]); ax.axis("off")
            words = _highlight_words(store[k]["caption"], per[k]["hallucinated_words"] + per[k]["hallucinated"])
            _draw_caption(ax, f"{lab}  [{len(per[k]['hallucinated'])} hallucinated]", col, words, wchars)
    return save(fig, "fig13_qualitative")


def make_all_figures(R):
    for fn in [fig_pipeline, fig_pairs, fig_training, fig_main_chair, fig_tradeoff, fig_pope,
               fig_length, fig_lambda, fig_categories, fig_mme, fig_bootstrap, fig_reward_validity,
               fig_qualitative]:
        try:
            fn(R)
        except Exception as e:            # one failing figure never blocks the others
            import traceback; print(f"[figure] {fn.__name__} failed: {e}"); traceback.print_exc()
    jdump(FIGS, P["pubdata"] / "figure_index.json")
    return FIGS


## 13 · Journal tables (LaTeX booktabs + CSV)

In [ ]:
# ============================================================================
# 13) JOURNAL TABLES — LaTeX (booktabs) + CSV, generated from results_master.json
# ============================================================================
import csv

TABS = {}


def fmt(v, sd=None, n=1, d=1):
    if v is None:
        return "--"
    s = f"{v:.{d}f}"
    if sd is not None and n and n > 1:
        s += f" $\\pm$ {sd:.{d}f}"
    return s


def _tex(x):
    """Unicode -> LaTeX for table cells (labels are shared with figures)."""
    x = str(x)
    if "$" not in x and "\\" not in x:
        x = x.replace("_", "\\_")
    return (x.replace("\u03bb", "$\\lambda$").replace("\u03b1", "$\\alpha$")
            .replace("alpha=0", "$\\alpha{=}0$").replace("lambda = ", "$\\lambda$ = "))


def write_table(name, caption, label, header, rows, colspec=None, notes=None):
    colspec = colspec or ("l" + "c" * (len(header) - 1))
    tex = ["\\begin{table}[t]", "\\centering", "\\small", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
           f"\\begin{{tabular}}{{{colspec}}}", "\\toprule", " & ".join(header) + " \\\\", "\\midrule"]
    for r in rows:
        tex.append("\\midrule" if r == "MID" else " & ".join(_tex(x) for x in r) + " \\\\")
    tex += ["\\bottomrule", "\\end{tabular}"]
    if notes:
        tex.append(f"\\par\\smallskip\\footnotesize\\raggedright {notes}")
    tex.append("\\end{table}")
    (P["tab"] / f"{name}.tex").write_text("\n".join(tex) + "\n")
    with open(P["tab"] / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow([re.sub(r"[\\$]", "", h) for h in header])
        for r in rows:
            if r != "MID":
                w.writerow([re.sub(r"\$\\pm\$", "±", str(x)).replace("\\", "") for x in r])
    TABS[name] = {"caption": caption, "label": label}
    print("  table:", name)


def _g(R, g, k, d=1):
    v = R["groups"].get(g, {}).get(k)
    return fmt(v["mean"], v["sd"], v["n"], d) if v else "--"


def table_setup(R):
    t, h = C["train"], C["har"]
    rows = [["Base model", "LLaVA-1.5-7B (\\texttt{llava-hf/llava-1.5-7b-hf}), bf16"],
            ["Detector", f"Grounding DINO Swin-T; box/text thresholds {C['detector']['box_threshold']}/"
                         f"{C['detector']['text_threshold']}; keep score {C['detector']['keep_score']}"],
            ["Entity matcher", f"all-MiniLM-L6-v2, cosine threshold $\\tau$ = {h['tau']}"],
            ["Reward", f"$R=\\alpha S-\\lambda H$, $\\alpha$ = {h['alpha']}, $\\lambda$ = {h['lam']}; "
                       f"min. pair margin {h['min_margin']}"],
            ["On-policy sampling", f"k = {C['sampling']['k']}, T = {C['sampling']['temperature']}, "
                                   f"top-p = {C['sampling']['top_p']}, max {C['max_new_tokens_caption']} tokens"],
            ["Training images / round", f"{C['n_train']:,} (Karpathy train split, train2014 only)"],
            ["LoRA", f"r = {t['lora_r']}, $\\alpha$ = {t['lora_alpha']}, dropout {t['lora_dropout']}; "
                     "language-model linear layers only"],
            ["DPO", f"$\\beta$ = {t['beta']}, lr = {t['lr']:g}, {t['epochs']} epochs, batch "
                    f"{t['per_device_bs']}$\\times${t['grad_accum']}, linear decay"],
            ["Seeds", ", ".join(map(str, C["seeds"]))],
            ["Hardware", f"1$\\times$ {ENV.get('gpu', 'GPU')}"]]
    write_table("tab01_setup", "Experimental configuration.", "tab:setup",
                ["Component", "Setting"], rows, colspec="lp{11cm}")


def table_chair(R):
    gs = [g for g in ["base", "base_vcd", "har_main", "ours_vcd", "iter2"] if g in R["groups"]
          and "CHAIRi" in R["groups"][g]]
    rows = [[SHORT[g], _g(R, g, "CHAIRs"), _g(R, g, "CHAIRi"), _g(R, g, "Recall"), _g(R, g, "avg_words", 1)]
            for g in gs]
    write_table("tab02_chair", f"Object hallucination on {C['n_chair']} COCO val2014 images "
                "(official CHAIR; greedy decoding, max "
                f"{C['max_new_tokens_caption']} new tokens). Mean $\\pm$ SD over training seeds where applicable.",
                "tab:chair", ["Method", "CHAIR$_S\\downarrow$", "CHAIR$_I\\downarrow$", "Recall$\\uparrow$", "Words"], rows)


def table_pope(R):
    gs = [g for g in ["base", "base_vcd", "har_main", "ours_vcd", "iter2"] if g in R["groups"]
          and any(k.startswith("POPE_") for k in R["groups"][g])]
    rows = []
    for s in C["pope_settings"]:
        for g in gs:
            rows.append([s.capitalize() if g == gs[0] else "", SHORT[g],
                         *[_g(R, g, f"POPE_{s}_{k}") for k in ("Accuracy", "Precision", "Recall", "F1", "Yes_ratio")]])
        rows.append("MID")
    write_table("tab03_pope", "POPE results on COCO (3{,}000 questions per setting).", "tab:pope",
                ["Setting", "Method", "Acc.", "Prec.", "Rec.", "F1", "Yes (\\%)"], rows[:-1], colspec="llccccc")


def table_mme(R):
    gs = [g for g in ["base", "har_main", "iter2"] if g in R["groups"] and "MME_perception" in R["groups"][g]]
    if not gs:
        return
    rows = [[SHORT[g], _g(R, g, "MME_hallucination"), _g(R, g, "MME_perception"), _g(R, g, "MME_cognition")]
            for g in gs]
    write_table("tab04_mme", "MME scores (acc + acc+). Hallucination subset = existence, count, position, colour "
                "(max 800); perception max 2000; cognition max 800.", "tab:mme",
                ["Method", "Halluc. subset", "Perception", "Cognition"], rows)


def table_ablations(R):
    order = ["har_main", "abl_coco80", "abl_h_only", "lam1", "lam4", "ctrl_length", "ctrl_random", "iter2"]
    rows = [[SHORT["base"], "--", *[f"{R['policies']['base']['metrics'].get(k, float('nan')):.1f}"
                                    for k in ("CHAIRs", "CHAIRi", "Recall", "avg_words")],
             fmt(R["policies"]["base"]["metrics"].get("POPE_avg_F1"))], "MID"]
    for g in order:
        name = f"{g}_s{S0}"
        if name not in R["policies"]:
            continue
        m = R["policies"][name]["metrics"]
        npairs = R["pairs"].get(g if g != "iter2" else "iter2", {}).get("n_pairs", "--")
        rows.append([SHORT[g], f"{npairs:,}" if isinstance(npairs, int) else npairs,
                     *[fmt(m.get(k)) for k in ("CHAIRs", "CHAIRi", "Recall", "avg_words")], fmt(m.get("POPE_avg_F1"))])
    write_table("tab05_ablations", f"Ablations and matched controls (training seed {S0}; identical "
                "samples, schedule and evaluation; controls use exactly the images retained by HAR).",
                "tab:ablations", ["Variant", "Pairs", "CHAIR$_S$", "CHAIR$_I$", "Recall", "Words", "POPE F1"], rows)


def table_tests(R):
    rows = []
    for name, t in R["tests"].items():
        d, ci = t["delta"], t["ci95"]
        rows.append([LABELS.get(R["policies"][name]["group"], name) + (f" (s{R['policies'][name]['seed']})"
                     if R["policies"][name]["seed"] is not None else ""),
                     f"{d['CHAIRi']:+.2f} [{ci['CHAIRi'][0]:+.2f}, {ci['CHAIRi'][1]:+.2f}]",
                     f"{d['CHAIRs']:+.2f} [{ci['CHAIRs'][0]:+.2f}, {ci['CHAIRs'][1]:+.2f}]",
                     f"{d['Recall']:+.2f}", f"{t['wilcoxon_p_halluc_count']:.2g}",
                     f"{t.get('wilcoxon_p_holm', float('nan')):.2g}", f"{t['mcnemar_chairs']['p']:.2g}"])
    write_table("tab06_significance", "Paired comparison with the base model on the same images: change "
                "with 95\\% bootstrap CI (" + f"{C['n_bootstrap']:,}" + " resamples), Wilcoxon signed-rank test "
                "on per-image hallucinated mentions (raw and Holm-adjusted), exact McNemar test on CHAIR$_S$.",
                "tab:tests", ["Policy", "$\\Delta$CHAIR$_I$ [CI]", "$\\Delta$CHAIR$_S$ [CI]", "$\\Delta$Recall",
                              "$p_W$", "$p_{W,\\mathrm{Holm}}$", "$p_{McN}$"], rows, colspec="lcccccc")


def table_pairs(R):
    rows = [[SHORT.get(k, k), f"{v['n_pairs']:,}", fmt(v.get("H_chosen"), d=3), fmt(v.get("H_rejected"), d=3),
             fmt(v.get("words_chosen"), d=0), fmt(v.get("words_rejected"), d=0),
             fmt(100 * v["frac_chosen_shorter"], d=1) if "frac_chosen_shorter" in v else "--"]
            for k, v in R["pairs"].items() if v.get("n_pairs")]
    write_table("tab07_pairs", "Preference-pair statistics. $H$ is always the caption-conditioned hallucination "
                "rate, so variants are comparable; the last column is the share of pairs whose preferred "
                "caption is the shorter one.", "tab:pairs",
                ["Variant", "Pairs", "$H$ chosen", "$H$ rejected", "Words ch.", "Words rej.", "Ch. shorter (\\%)"], rows)


def table_compute(R):
    led = read_ledger(); agg = {}
    for r in led:
        if r["status"] != "ok":
            continue
        k = ("DPO training" if r["stage"].startswith("train:") else "Evaluation (all policies)"
             if r["stage"].startswith("eval:") else r["stage"])
        a = agg.setdefault(k, [0.0, 0.0]); a[0] += r["seconds"] / 3600; a[1] = max(a[1], r["peak_gpu_gb"] or 0)
    rows = [[k.replace("_", " "), f"{v[0]:.2f}", f"{v[1]:.1f}"] for k, v in sorted(agg.items(), key=lambda kv: -kv[1][0])]
    rows += ["MID", ["Total", f"{sum(v[0] for v in agg.values()):.2f}", ""]]
    write_table("tab08_compute", f"Measured wall-clock cost on one {ENV.get('gpu', 'GPU')}.", "tab:compute",
                ["Stage", "GPU hours", "Peak memory (GB)"], rows)


def table_seeds(R):
    rows = [[n, v["seed"], *[fmt(v["metrics"].get(k)) for k in ("CHAIRs", "CHAIRi", "Recall", "avg_words", "POPE_avg_F1")]]
            for n, v in R["policies"].items() if v["group"] == "har_main"]
    write_table("tab09_per_seed", "Per-seed results of the main model (supplementary).", "tab:seeds",
                ["Run", "Seed", "CHAIR$_S$", "CHAIR$_I$", "Recall", "Words", "POPE F1"], rows)


def make_all_tables(R):
    for fn in [table_setup, table_chair, table_pope, table_mme, table_ablations, table_tests,
               table_pairs, table_compute, table_seeds]:
        try:
            fn(R)
        except Exception as e:
            import traceback; print(f"[table] {fn.__name__} failed: {e}"); traceback.print_exc()
    jdump(TABS, P["pubdata"] / "table_index.json")
    return TABS


---
# Run the study
Each stage below is resumable. Run them in order; after a disconnect, re-run the setup cells (0–13) and continue.

### Run A · Base model evaluation

In [ ]:
# ============================================================================
# RUN A — base model evaluation (CHAIR, POPE x3, MME). Resumable.
# ============================================================================
BASE_METRICS = stage_base_eval()
print(json.dumps({k: v for k, v in BASE_METRICS.get("chair", {}).items()}, indent=1))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  base captions 80/500 (0.5 min)
  base captions 160/500 (1.1 min)
  base captions 240/500 (1.7 min)
  base captions 320/500 (2.2 min)
  base captions 400/500 (2.7 min)
  base captions 480/500 (3.3 min)
  base captions 500/500 (3.5 min)
[ledger] eval:base:chair: 3.5 min, peak 19.27 GB (ok)
[ledger] eval:base:pope_random: 3.0 min, peak 24.68 GB (ok)
[ledger] eval:base:pope_popular: 3.1 min, peak 24.68 GB (ok)
[ledger] eval:base:pope_adversarial: 3.1 min, peak 24.68 GB (ok)
[ledger] eval:base:mme: 4.1 min, peak 25.22 GB (ok)
{
 "CHAIRs": 55.8,
 "CHAIRi": 16.451169188445668,
 "Recall": 76.44787644787645,
 "avg_words": 92.716,
 "n_captions": 500,
 "n_object_mentions": 3635,
 "n_hallucinated_mentions": 598,
 "avg_tokens": 107.568,
 "frac_finished": 1.0
}


### Run B · Round-1 on-policy data and all preference-pair variants
Prints pair statistics, including `frac_chosen_shorter` — the share of pairs where HAR prefers the shorter caption. If it is far above 0.5, the length control (Tier 1) is essential to the paper's argument.

In [ ]:
# ============================================================================
# RUN B — round-1 on-policy data: k samples/image -> Grounding DINO -> rewards -> pairs
# ============================================================================
SAMPLES_R1, DETS_R1, SCORES_R1 = stage_round_data(1, D["train_r1"], {"name": "base", "adapters": []})
PAIRS = {}
for e in [x for x in ACTIVE if x.get("round", 1) == 1]:
    PAIRS[e["name"]], st = stage_pairs(e, D["train_r1"], SCORES_R1)
jdump({k: jload(P["cache"] / "pairs" / f"{k}.json")["stats"] for k in PAIRS}, P["pubdata"] / "pair_stats.json")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  sampled 80/5000 images (1.4 min, ~85 min left)
  sampled 160/5000 images (2.8 min, ~84 min left)
  sampled 240/5000 images (4.1 min, ~81 min left)
  sampled 320/5000 images (5.5 min, ~80 min left)
  sampled 400/5000 images (6.9 min, ~79 min left)
  sampled 480/5000 images (8.2 min, ~77 min left)
  sampled 560/5000 images (9.5 min, ~75 min left)
  sampled 640/5000 images (10.9 min, ~74 min left)
  sampled 720/5000 images (12.2 min, ~73 min left)
  sampled 800/5000 images (13.5 min, ~71 min left)
  sampled 880/5000 images (14.9 min, ~70 min left)
  sampled 960/5000 images (16.2 min, ~68 min left)
  sampled 1040/5000 images (17.6 min, ~67 min left)
  sampled 1120/5000 images (18.9 min, ~66 min left)
  sampled 1200/5000 images (20.3 min, ~64 min left)
  sampled 1280/5000 images (21.6 min, ~63 min left)
  sampled 1360/5000 images (23.0 min, ~62 min left)
  sampled 1440/5000 images (24.4 min, ~60 min left)
  sampled 1520/5000 images (25.8 min, ~59 min left)
  sampled 1600/5000 images (27.1

/content/aavlm_local/python_packages/transformers/models/grounding_dino/processing_grounding_dino.py:98: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


  detection[cc] 8/5000 (0.0 min)
  detection[cc] 408/5000 (1.2 min)
  detection[cc] 808/5000 (2.4 min)
  detection[cc] 1208/5000 (3.6 min)
  detection[cc] 1608/5000 (4.8 min)
  detection[cc] 2008/5000 (6.0 min)
  detection[cc] 2408/5000 (7.1 min)
  detection[cc] 2808/5000 (8.3 min)
  detection[cc] 3208/5000 (9.5 min)
  detection[cc] 3608/5000 (10.7 min)
  detection[cc] 4008/5000 (11.8 min)
  detection[cc] 4408/5000 (13.0 min)
  detection[cc] 4808/5000 (14.2 min)
[ledger] detection_cc_r1: 14.8 min, peak 33.38 GB (ok)
  detection[coco80] 8/5000 (0.0 min)
  detection[coco80] 408/5000 (1.2 min)
  detection[coco80] 808/5000 (2.4 min)
  detection[coco80] 1208/5000 (3.6 min)
  detection[coco80] 1608/5000 (4.8 min)
  detection[coco80] 2008/5000 (5.9 min)
  detection[coco80] 2408/5000 (7.1 min)
  detection[coco80] 2808/5000 (8.3 min)
  detection[coco80] 3208/5000 (9.4 min)
  detection[coco80] 3608/5000 (10.6 min)
  detection[coco80] 4008/5000 (11.8 min)
  detection[coco80] 4408/5000 (13.0 min)


### Run C · Train and evaluate every active experiment
Main model (all seeds) first, then controls and ablations. `progress_report()` prints measured GPU-hours and a projection for the remaining runs.

In [ ]:
# ============================================================================
# RUN C — HAR-DPO training + evaluation for every active round-1 experiment.
# Order: main model (all seeds) first, then controls/ablations, so the most
# important results exist even if the session budget runs out. Re-run this
# cell after any disconnect: finished adapters/evaluations are skipped and an
# interrupted training run resumes from its last checkpoint.
# ============================================================================
for e in [x for x in ACTIVE if x.get("round", 1) == 1]:
    if e["name"] not in PAIRS:
        PAIRS[e["name"]] = jload(P["cache"] / "pairs" / f"{e['name']}.json")["pairs"]
    stage_train_and_eval(e, PAIRS[e["name"]])
    progress_report()


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[train] har_main_s42: trainable 39,976,960 / 7,103,404,032 (0.563%)


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
5,0.689600
10,0.687500
15,0.690500
20,0.695300
25,0.697700
30,0.695500
35,0.668200
40,0.669900
45,0.668700
50,0.668100


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

[ledger] train:har_main_s42: 89.6 min, peak 42.52 GB (ok)


... (62 outputs left out)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

[ledger] train:lam4_s42: 97.0 min, peak 42.52 GB (ok)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  lam4_s42 captions 80/500 (0.5 min)
  lam4_s42 captions 160/500 (1.0 min)
  lam4_s42 captions 240/500 (1.5 min)
  lam4_s42 captions 320/500 (2.0 min)
  lam4_s42 captions 400/500 (2.5 min)
  lam4_s42 captions 480/500 (3.1 min)
  lam4_s42 captions 500/500 (3.3 min)
[ledger] eval:lam4_s42:chair: 3.3 min, peak 19.36 GB (ok)
[ledger] eval:lam4_s42:pope_random: 3.0 min, peak 24.68 GB (ok)
[ledger] eval:lam4_s42:pope_popular: 3.0 min, peak 24.68 GB (ok)
[ledger] eval:lam4_s42:pope_adversarial: 3.0 min, peak 24.68 GB (ok)
GPU hours logged so far: 17.38
measured mean DPO run: 1.45 h  ->  remaining training ~1.5 h for 1 runs
trained: ['har_main_s42', 'har_main_s123', 'har_main_s456', 'ctrl_length_s42', 'abl_coco80_s42', 'abl_h_only_s42', 'ctrl_random_s42', 'lam1_s42', 'lam4_s42']
pending: ['iter2_s42']


### Run D · VCD baseline and Auto Align-VLM + VCD

In [ ]:
# ============================================================================
# RUN D — training-free decoding baseline (VCD) and its combination with ours
# ============================================================================
stage_decoding_baselines()


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  base_vcd captions 80/500 (1.0 min)
  base_vcd captions 160/500 (2.1 min)
  base_vcd captions 240/500 (3.1 min)
  base_vcd captions 320/500 (4.1 min)
  base_vcd captions 400/500 (5.2 min)
  base_vcd captions 480/500 (6.2 min)
  base_vcd captions 500/500 (6.7 min)
[ledger] eval:base_vcd:chair: 6.7 min, peak 25.55 GB (ok)
[ledger] eval:base_vcd:pope_random: 5.5 min, peak 35.58 GB (ok)
[ledger] eval:base_vcd:pope_popular: 5.5 min, peak 35.58 GB (ok)
[ledger] eval:base_vcd:pope_adversarial: 5.5 min, peak 35.58 GB (ok)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  ours_vcd captions 80/500 (1.0 min)
  ours_vcd captions 160/500 (2.1 min)
  ours_vcd captions 240/500 (3.1 min)
  ours_vcd captions 320/500 (4.0 min)
  ours_vcd captions 400/500 (5.0 min)
  ours_vcd captions 480/500 (5.9 min)
  ours_vcd captions 500/500 (6.3 min)
[ledger] eval:ours_vcd:chair: 6.3 min, peak 25.28 GB (ok)
[ledger] eval:ours_vcd:pope_random: 5.5 min, peak 35.58 GB (ok)
[ledger] eval:ours_vcd:pope_popular: 5.5 min, peak 35.58 GB (ok)
[ledger] eval:ours_vcd:pope_adversarial: 5.5 min, peak 35.58 GB (ok)


### Run E · (Tier 2) Second on-policy round

In [39]:
# ============================================================================
# RUN E — (Tier 2) second on-policy round: sample from the round-1 policy on
# NEW images, re-score with HAR, and train a new adapter on top of it.
# ============================================================================
it = [e for e in ACTIVE if e.get("round") == 2]
if it:
    e = it[0]
    ensure_images(D["sel"]["train_r2"], "train2014", "train_r2")
    r1_spec = policy_spec(next(x for x in EXPERIMENTS if x["name"] == e["init"]), S0)
    _, _, SCORES_R2 = stage_round_data(2, D["train_r2"], r1_spec)
    PAIRS[e["name"]], _ = stage_pairs(e, D["train_r2"], SCORES_R2)
    stage_train_and_eval(e, PAIRS[e["name"]])
else:
    print("Tier 2 not active: iterative round skipped.")


[images] fetching 4994 train2014 images for 'train_r2' ...
[images] train_r2: 5000 ready


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  sampled 80/5000 images (1.3 min, ~81 min left)
  sampled 160/5000 images (2.7 min, ~81 min left)
  sampled 240/5000 images (4.0 min, ~79 min left)
  sampled 320/5000 images (5.3 min, ~78 min left)
  sampled 400/5000 images (6.6 min, ~76 min left)
  sampled 480/5000 images (7.9 min, ~75 min left)
  sampled 560/5000 images (9.2 min, ~73 min left)
  sampled 640/5000 images (10.5 min, ~72 min left)
  sampled 720/5000 images (11.9 min, ~70 min left)
  sampled 800/5000 images (13.1 min, ~69 min left)
  sampled 880/5000 images (14.5 min, ~68 min left)
  sampled 960/5000 images (15.8 min, ~66 min left)
  sampled 1040/5000 images (17.1 min, ~65 min left)
  sampled 1120/5000 images (18.4 min, ~64 min left)
  sampled 1200/5000 images (19.7 min, ~62 min left)
  sampled 1280/5000 images (21.1 min, ~61 min left)
  sampled 1360/5000 images (22.3 min, ~60 min left)
  sampled 1440/5000 images (23.6 min, ~58 min left)
  sampled 1520/5000 images (25.0 min, ~57 min left)
  sampled 1600/5000 images (26.3

/content/aavlm_local/python_packages/transformers/models/grounding_dino/processing_grounding_dino.py:98: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


  detection[cc] 8/5000 (0.0 min)
  detection[cc] 408/5000 (1.1 min)
  detection[cc] 808/5000 (2.3 min)
  detection[cc] 1208/5000 (3.4 min)
  detection[cc] 1608/5000 (4.5 min)
  detection[cc] 2008/5000 (5.7 min)
  detection[cc] 2408/5000 (6.8 min)
  detection[cc] 2808/5000 (7.9 min)
  detection[cc] 3208/5000 (9.1 min)
  detection[cc] 3608/5000 (10.2 min)
  detection[cc] 4008/5000 (11.4 min)
  detection[cc] 4408/5000 (12.5 min)
  detection[cc] 4808/5000 (13.6 min)
[ledger] detection_cc_r2: 14.2 min, peak 33.33 GB (ok)
  scored 500/5000 images
  scored 1000/5000 images
  scored 1500/5000 images
  scored 2000/5000 images
  scored 2500/5000 images
  scored 3000/5000 images
  scored 3500/5000 images
  scored 4000/5000 images
  scored 4500/5000 images
  scored 5000/5000 images
[ledger] scoring_r2: 1.9 min, peak 13.4 GB (ok)
[pairs] iter2: {'n_pairs': 4478, 'skipped': 522, 'mean_margin': 0.59, 'H_chosen': 0.2205, 'H_rejected': 0.4815, 'S_chosen': 0.8201, 'S_rejected': 0.7521, 'words_chosen': 7

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

... (5 outputs left out)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


[ledger] train:iter2_s42: 88.6 min, peak 42.45 GB (ok)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  iter2_s42 captions 80/500 (0.5 min)
  iter2_s42 captions 160/500 (1.0 min)
  iter2_s42 captions 240/500 (1.5 min)
  iter2_s42 captions 320/500 (2.0 min)
  iter2_s42 captions 400/500 (2.5 min)
  iter2_s42 captions 480/500 (3.1 min)
  iter2_s42 captions 500/500 (3.3 min)
[ledger] eval:iter2_s42:chair: 3.3 min, peak 19.3 GB (ok)
[ledger] eval:iter2_s42:pope_random: 3.0 min, peak 24.68 GB (ok)
[ledger] eval:iter2_s42:pope_popular: 3.0 min, peak 24.68 GB (ok)
[ledger] eval:iter2_s42:pope_adversarial: 3.0 min, peak 24.68 GB (ok)
[ledger] eval:iter2_s42:mme: 4.1 min, peak 25.22 GB (ok)


### Run F · Analysis, figures, tables, publication bundle
Writes `publication/` (figures, tables, data, README) and a zip next to it. These files are the only inputs used to write the manuscript.

In [40]:
# ============================================================================
# RUN F — analysis, journal figures, tables, and the single publication bundle
# ============================================================================
RV = stage_reward_validity()
R = collect_results()
make_all_figures(R)
make_all_tables(R)

readme = ["# Auto Align-VLM v3 — publication bundle", "",
          f"Profile: {C['profile']} | config hash {config_hash(C)} | generated {datetime.datetime.now():%Y-%m-%d %H:%M}",
          "", "All numbers come from `data/results_master.json`, produced by the notebook from cached raw",
          "model outputs (`../eval/<policy>/`). Figures are vector PDF + PNG; tables are LaTeX + CSV.", "",
          "## Figures"] + [f"- `{Path(v['files'][0]).name}`" for v in FIGS.values()] + \
         ["", "## Tables"] + [f"- `{k}.tex` — {v['caption'][:90]}..." for k, v in TABS.items()] + \
         ["", "## Data", "- `results_master.json` — every metric, test and compute figure",
          "- `run_config.json` — configuration + library versions", "- `reward_validity.json`, "
          "`bootstrap_distributions.json`, `pair_stats.json`, `figure_index.json`, `table_index.json`"]
(P["pub"] / "README.md").write_text("\n".join(readme) + "\n")
shutil.copy(LEDGER, P["pubdata"] / "compute_ledger.jsonl")
bundle = shutil.make_archive(str(ROOT / f"AutoAlignVLM_v3_publication_{C['profile']}"), "zip", P["pub"])
print("publication bundle:", bundle)
print("headline:", {g: {k: R["groups"][g][k]["mean"] for k in ("CHAIRs", "CHAIRi", "Recall")
                        if k in R["groups"][g]} for g in ("base", "har_main") if g in R["groups"]})


/content/aavlm_local/python_packages/transformers/models/grounding_dino/processing_grounding_dino.py:98: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


  detection[cc] 8/500 (0.0 min)
  detection[cc] 408/500 (1.0 min)
[ledger] reward_validity_detection: 1.3 min, peak 18.58 GB (ok)
results_master.json: 13 policies, 12 paired tests, 21.553 GPU-h logged
  figure: fig01_pipeline
  figure: fig02_preference_pairs
  figure: fig03_training_dynamics
  figure: fig04_chair_main
  figure: fig05_tradeoff
  figure: fig06_pope
  figure: fig07_length_analysis
  figure: fig08_lambda_sensitivity
  figure: fig09_categories
  figure: fig10_mme
  figure: fig11_bootstrap
  figure: fig12_reward_validity
  figure: fig13_qualitative
  table: tab01_setup
  table: tab02_chair
  table: tab03_pope
  table: tab04_mme
  table: tab05_ablations
  table: tab06_significance
  table: tab07_pairs
  table: tab08_compute
  table: tab09_per_seed
publication bundle: /content/drive/MyDrive/ResearchforPub/auto-align-vlm-execution/auto-align-vlm_v3/AutoAlignVLM_v3_publication_full.zip
headline: {'base': {'CHAIRs': 55.8, 'CHAIRi': 16.451169188445668, 'Recall': 76.44787644787645}

---
### Output layout on Drive
```
auto-align-vlm_v3/
├── cache/        selections, POPE, annotations, images/*.tar, samples, detections, scores, pairs/
├── runs/         <experiment>_s<seed>/ final_adapter, train_pairs.json, training_log.json, train_meta.json
├── eval/         <policy>/ chair_captions.jsonl, chair_per_image.json, pope_*.jsonl, mme.jsonl, metrics.json
├── logs/         compute_ledger.jsonl
├── publication/  figures/ (PDF+PNG) · tables/ (TEX+CSV) · data/ (results_master.json, …) · README.md
└── AutoAlignVLM_v3_publication_full.zip
```
Send the zip (or `publication/data/results_master.json` plus the figures) back, and the manuscript will be written strictly from those numbers.